# DeepVoice: Do Fake-Voice Detectors Learn Fakeness, or Just the Dataset?

**CMPS 261 Introduction to Machine Learning, AUB, Fall 2026.** Hashem Awad, Ahmed Faleh, Ali Sleiman.
Report: submitted on Moodle. Code: https://github.com/HashemAwad/deepvoice

**Question.** What do fake-voice detectors learn, and how much of their performance survives when the
data change? We train a ladder of detectors on ASVspoof 2019 LA (rung 0: silence/duration only;
rung 1: tuned classical models on LFCC; rung 2: frozen XLS-R-300M + logistic regression), test them on
unseen attacks and on In-the-Wild, and probe shortcuts, calibration and failures.

**Rules that every cell follows** (see `CLAUDE.md`): fit on train only; choose settings and thresholds
on dev; eval and In-the-Wild were used only after `hypotheses.md` was frozen and hashed (commit
`1d896cd`, sha256 in the README); primary metric EER with a speaker-clustered bootstrap 95% CI;
features computed once and cached; seed 0 and a config file for every run.

**How to run.** `./scripts/setup.sh --dl`, `./scripts/download_data.sh` (about 16 GB), then
`uv run --extra dl python -m deepvoice.pipeline` (builds all caches; about 3 h from scratch, seconds
when cached), then run this notebook top to bottom. Each section below is one of the repository's
notebooks 00-06, unchanged. The reusable code they import (audio loader, features, XLS-R embedding,
metrics, models, final-model CLI) lives in `src/deepvoice/` and is reproduced in full in the
**Appendix** at the end of this notebook.

## 0. Build (or load) every cached feature set

In [ ]:
from pathlib import Path

from deepvoice.pipeline import build_all

build_all("../data" if Path.cwd().name == "notebooks" else "data")

---
# 1. Data audit

Verified split counts, class balance, duration and silence by class (train and dev). Figure 1 of the report. *(Source: `notebooks/00_audit.ipynb`.)*

## 00 Data audit (ASVspoof 2019 LA, train + dev only)

Class balance, duration and leading/trailing silence by class. Eval and In-the-Wild are not
touched here (they stay locked until `hypotheses.md` is frozen). Silence comes from the
energy VAD in `deepvoice.silence` (frame = speech if within 35 dB of the clip's loudest frame).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# protocol table: one row per clip (file, speaker, attack, label, split, path)
proto = pd.read_parquet(ROOT / "data/protocols.parquet")
# cached silence/duration features of train and dev (eval stays untouched here),
# joined with labels; label 1 = fake, 0 = real
df = pd.concat(
    [pd.read_parquet(ROOT / f"data/features/silence_{s}.parquet").assign(split=s) for s in ["train", "dev"]]
).merge(proto[["file", "label", "attack", "speaker"]], on="file")
df["cls"] = df["label"].map({0: "real", 1: "fake"})

## Protocol counts (all splits; labels only, no audio)

In [ ]:
from deepvoice.data import summarize

counts = summarize(proto)
counts.to_csv(ROOT / "results/protocol_counts.csv", index=False)
counts

## Class balance

In [ ]:
bal = df.groupby(["split", "cls"], sort=False).size().unstack()
bal["fake_share"] = (bal["fake"] / bal.sum(axis=1)).round(3)
bal

## Duration and silence by class (median and interquartile range)

In [ ]:
# median and interquartile range (25th-75th percentile) of each feature, per split and class;
# a big real-vs-fake difference in silence would be a shortcut a model could exploit
rows = []
for (split, cls), g in df.groupby(["split", "cls"], sort=False):
    r = {"split": split, "class": cls, "n": len(g)}
    for c in ["duration", "lead_sil", "trail_sil", "sil_ratio", "energy_max"]:
        r[c + "_median"] = round(g[c].median(), 3)
        r[c + "_iqr"] = f"{g[c].quantile(.25):.2f}-{g[c].quantile(.75):.2f}"
    rows.append(r)
audit = pd.DataFrame(rows)
audit.to_csv(ROOT / "results/audit_by_class.csv", index=False)
audit.T

In [ ]:
df[df.split == "train"].groupby("attack")[["duration", "lead_sil", "trail_sil", "sil_ratio"]].median().round(3)

## Figure for the progress report

In [ ]:
REAL, FAKE = "#2a78d6", "#eb6834"  # categorical slots 1-2 (colour-blind checked)
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1, 2.4, 1.3]})

# (a) class balance
for i, split in enumerate(["train", "dev"]):
    r, f = bal.loc[split, "real"], bal.loc[split, "fake"]
    ax[0].barh(i, r, color=REAL, height=0.6)
    ax[0].barh(i, f, left=r + 300, color=FAKE, height=0.6)
    ax[0].text(r + f + 900, i, f"{f / (r + f):.0%} fake", va="center", color=INK)
ax[0].set_yticks([0, 1], ["train", "dev"]); ax[0].invert_yaxis()
ax[0].set_xlim(0, 33000); ax[0].set_xlabel("clips")
ax[0].set_title("(a) Class balance", loc="left", color=INK)

# (b) leading silence: real vs each training attack, train and dev side by side
groups = ["bonafide", "A01", "A02", "A03", "A04", "A05", "A06"]
for j, split in enumerate(["train", "dev"]):
    d = df[df.split == split]
    data = [d.loc[d.attack == g, "lead_sil"] for g in groups]
    pos = [k * 3 + j for k in range(len(groups))]
    bp = ax[1].boxplot(data, positions=pos, widths=0.8, showfliers=False, patch_artist=True,
                       medianprops={"color": INK, "linewidth": 1.2})
    for patch, g in zip(bp["boxes"], groups):
        patch.set(facecolor=REAL if g == "bonafide" else FAKE, edgecolor="white",
                  alpha=1.0 if split == "train" else 0.55)
ax[1].set_xticks([k * 3 + 0.5 for k in range(len(groups))], ["real"] + groups[1:])
ax[1].set_ylabel("leading silence (s)"); ax[1].grid(axis="y", color=GRID, linewidth=0.6)
ax[1].set_title("(b) Leading silence by attack (left box train, right box dev)", loc="left", color=INK)

# (c) duration by class
for cls, col in [("real", REAL), ("fake", FAKE)]:
    ax[2].hist(df.loc[df.cls == cls, "duration"], bins=60, range=(0, 10), density=True,
               histtype="step", linewidth=2, color=col, label=cls)
ax[2].set_xlabel("duration (s)"); ax[2].set_yticks([])
ax[2].legend(frameon=False); ax[2].set_title("(c) Clip duration (train+dev)", loc="left", color=INK)

fig.tight_layout()
(ROOT / "results/figures").mkdir(exist_ok=True)
fig.savefig(ROOT / "results/figures/audit_silence.png", dpi=200)
fig.savefig(ROOT / "results/figures/audit_silence.pdf")

---
# 2. E2: feature engineering

Feature-group, MFCC-vs-LFCC, coefficient, delta and pooling ablations with fixed untuned models (dev only). *(Source: `notebooks/01_features.ipynb`.)*

## 01 E2: feature engineering ablation (train -> dev)

Fixed, untuned models: logistic regression (C=1) and random forest (300 trees), seed 0. Every
setting is trained on train and scored on **dev** with a speaker-clustered bootstrap 95% CI.
Eval is not used here. Results: `results/e2.csv`; figure: `results/figures/e2_features.png`.

Groups: S = silence/duration/energy (9), M = MFCC, L = LFCC, P = spectral (6 descriptors).
Unless stated otherwise M and L use 20 coefficients + delta + delta-delta, pooled mean+std.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.features import load_xy
from deepvoice.models import run

# E2 uses fixed, untuned models so that only the features change between runs
MODELS = {"logreg": {"C": 1.0}, "rf": {"n_estimators": 300, "n_jobs": 6}}
# short names: S = silence/duration, M = MFCC, L = LFCC, P = spectral statistics
NAMES = {"silence": "S", "mfcc": "M", "lfcc": "L", "spec": "P"}
# pooling: how frame-level coefficients are summarised into one vector per clip
MS, MSMM, MSPCT = ("mean", "std"), ("mean", "std", "min", "max"), ("mean", "std", "p10", "p50", "p90")


# readable name of a feature setting, used as its id in results/e2.csv
def label(groups, n_coef=20, deltas=2, stats=MS):
    if len(groups) == 1 and groups[0] in ("mfcc", "lfcc"):
        pool = {MS: "meanstd", MSMM: "meanstd_minmax", MSPCT: "meanstd_pct"}[stats]
        return f"{groups[0]}{n_coef}_d{deltas}_{pool}"
    return "+".join(NAMES[g] for g in groups)


# every feature setting to compare, grouped into families (the ablations)
settings = {}
def add(family, groups, **kw):
    settings.setdefault(label(groups, **kw), (family, groups, kw))

ALL = ["silence", "mfcc", "lfcc", "spec"]
# each group alone, then all groups minus one, then all together
for g in ALL:
    add("group: one", [g])
for g in ALL:
    add("group: drop one", [x for x in ALL if x != g])
add("group: all", ALL)
# number of coefficients x number of deltas, and richer pooling
for g in ["mfcc", "lfcc"]:
    for n in [13, 20, 40]:
        for d in [0, 1, 2]:
            add("coefficients x deltas", [g], n_coef=n, deltas=d)
    for st in [MSMM, MSPCT]:
        add("pooling", [g], stats=st)
print(len(settings), "feature settings x", len(MODELS), "models")

In [ ]:
RERUN = False  # True recomputes settings already in results/e2.csv
out = ROOT / "results/e2.csv"
# skip settings already in the results file (features are cached, results too)
done = set(pd.read_csv(out)["config"]) if out.exists() and not RERUN else set()
for fs, (family, groups, kw) in settings.items():
    todo = [m for m in MODELS if f"e2_{m}_{fs}_s0" not in done]
    if not todo:
        continue
    # load cached features: fit on train, score on dev only
    tr, dv = load_xy("train", groups, **kw), load_xy("dev", groups, **kw)
    # the first 4 columns are file, label, speaker, attack; the rest are features
    X = list(tr.columns[4:])
    for m in todo:
        # run() fits, scores, computes EER with a speaker-bootstrap CI and saves config + CSV row
        run(1, m, X, tr, [("asvspoof", "dev", dv)], params=MODELS[m], experiment="e2", feature_set=fs)
    print(f"{fs:28s} {len(X):4d} features done")

## Results (dev EER %, 95% CI)

In [ ]:
# turn the long results file into one table: setting x model -> dev EER [95% CI]
r = pd.read_csv(out)
r["setting"] = r["model"].str.split(":").str[1]
r["model"] = r["model"].str.split(":").str[0]
r["family"] = r["setting"].map(lambda s: settings[s][0])
# feature count read back from each run's saved config file
r["n_features"] = r["config"].map(lambda c: len(pd.read_json(ROOT / f"results/configs/{c}.json", typ="series")["features"]))
r["dev EER %"] = r.apply(lambda x: f"{100*x.eer:.2f} [{100*x.ci_low:.2f}, {100*x.ci_high:.2f}]", axis=1)
table = r.pivot_table(index=["family", "setting", "n_features"], columns="model", values="dev EER %", aggfunc="first")
table.to_csv(ROOT / "results/e2_table.csv")
table

## Figure

In [ ]:
LR, RF = "#1baf7a", "#6250d6"  # model colours (validated palette); real/fake keep blue/orange
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})

# (a) feature groups: alone, all, all-but-one
order = ["S", "P", "mfcc20_d2_meanstd", "lfcc20_d2_meanstd", "S+M+L+P",
         "M+L+P", "S+L+P", "S+M+P", "S+M+L"]
names = ["S only", "P only", "M only", "L only", "all (S+M+L+P)",
         "all - S", "all - M", "all - L", "all - P"]
for k, (m, col) in enumerate([("logreg", LR), ("rf", RF)]):
    d = r[r.model == m].set_index("setting").loc[order]
    y = [i + (k - 0.5) * 0.38 for i in range(len(order))]
    ax[0].barh(y, 100 * d.eer, height=0.36, color=col, label={"logreg": "logistic regression", "rf": "random forest"}[m])
    ax[0].errorbar(100 * d.eer, y, xerr=[100 * (d.eer - d.ci_low), 100 * (d.ci_high - d.eer)],
                   fmt="none", ecolor=INK, elinewidth=0.8, capsize=2)
ax[0].set_yticks(range(len(order)), names); ax[0].invert_yaxis()
ax[0].set_xscale("log"); ax[0].set_xlabel("dev EER % (log scale; bars = 95% CI)")
ax[0].grid(axis="x", color=GRID, linewidth=0.6); ax[0].legend(frameon=False, loc="lower right")
ax[0].set_title("(a) Feature groups", loc="left", color=INK)

# (b) MFCC vs LFCC by number of coefficients (with delta + delta-delta)
for m, col in [("logreg", LR), ("rf", RF)]:
    for g, ls, mk in [("mfcc", "--", "o"), ("lfcc", "-", "s")]:
        d = r[(r.model == m)].set_index("setting").loc[[f"{g}{n}_d2_meanstd" for n in [13, 20, 40]]]
        ax[1].plot([13, 20, 40], 100 * d.eer, ls=ls, marker=mk, ms=6, lw=2, color=col)
        ax[1].text(41.5, 100 * d.eer.iloc[-1], f"{g.upper()} {'LR' if m == 'logreg' else 'RF'}",
                   va="center", color=INK, fontsize=8)
ax[1].set_yscale("log"); ax[1].set_xticks([13, 20, 40]); ax[1].set_xlim(11, 48)
ax[1].set_xlabel("number of coefficients (+ delta + delta-delta)"); ax[1].set_ylabel("dev EER % (log scale)")
ax[1].grid(axis="y", color=GRID, linewidth=0.6)
ax[1].set_title("(b) MFCC (dashed) vs LFCC (solid)", loc="left", color=INK)
from matplotlib.ticker import FuncFormatter, NullFormatter

for a, axis in [(ax[0], "x"), (ax[1], "y")]:
    getattr(a, f"{axis}axis").set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    getattr(a, f"{axis}axis").set_minor_formatter(NullFormatter())
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e2_features.png", dpi=200)
fig.savefig(ROOT / "results/figures/e2_features.pdf")

---
# 3. E3: tuning, overfitting and split sensitivity

Random search on LFCC-40, selection on dev vs speaker-grouped CV, gap table and learning curves. *(Source: `notebooks/02_baselines.ipynb`.)*

## 02 E3: tuning, overfitting and split sensitivity (train -> dev)

Fixed features: LFCC, 40 coefficients + delta + delta-delta, mean+std (240 columns; best LFCC setting in E2). Seed 0.
Models: logistic regression, random forest, HistGradientBoosting (all class-balanced).

1. **Random search** (30 settings per model). Each setting is scored on dev and by
   speaker-grouped 5-fold CV on train. Full table: `results/e3_search.csv`.
2. **Selection** by dev and, separately, by grouped CV (split sensitivity), plus untuned defaults.
3. **Gap table**: train vs grouped-CV vs dev EER (eval is added later, once per final model).
4. **Learning curves**: dev-selected settings at 5/10/25/50/100% of train.

Final rows with 95% CIs: `results/e3.csv`. Figure: `results/figures/e3_tuning.png`.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import loguniform

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.features import load_xy
from deepvoice.models import random_search, run

# the feature set chosen in E2 (on dev): LFCC, 40 coefficients, deltas and delta-deltas,
# mean and std pooling
FS = {"groups": ("lfcc",), "n_coef": 40, "deltas": 2, "stats": ("mean", "std")}
train, dev = load_xy("train", **FS), load_xy("dev", **FS)
# feature columns (the first 4 columns are file, label, speaker, attack)
X = list(train.columns[4:])
print(train.shape, dev.shape, len(X), "features")

## 1. Random search

In [ ]:
# random-search spaces per model: (hyperparameter distributions, fixed settings);
# loguniform samples on a log scale, for parameters that span orders of magnitude
SPACES = {
    "logreg": ({"C": loguniform(1e-4, 1e2), "penalty": ["l1", "l2"]}, {"solver": "liblinear"}),
    "rf": ({"n_estimators": [100, 200, 400], "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 2, 5, 10], "max_features": ["sqrt", "log2", 0.3]}, {"n_jobs": 6}),
    "hgb": ({"learning_rate": loguniform(0.02, 0.3), "max_depth": [None, 3, 5, 8],
             "l2_regularization": loguniform(1e-3, 10), "max_iter": [100, 200, 400]},
            {"early_stopping": False}),
}
# untuned settings, kept as the reference point for 'how much did tuning help'
DEFAULTS = {"logreg": {"C": 1.0}, "rf": {"n_estimators": 300, "n_jobs": 6}, "hgb": {}}

search_csv = ROOT / "results/e3_search.csv"
if search_csv.exists():  # computed once; delete the file on purpose to redo the search
    search = pd.read_csv(search_csv)
else:
    # 30 random settings per model; each one gets a dev EER and a speaker-grouped
    # 5-fold CV EER on train (no speaker in both the fit and the validation fold)
    search = pd.concat([random_search(m, space, train, dev, X, n_iter=30, seed=0, fixed=fixed)
                        for m, (space, fixed) in SPACES.items()], ignore_index=True)
    search.to_csv(search_csv, index=False, float_format="%.6g")
search.groupby("model")[["dev_eer", "cv_eer", "fit_time"]].describe().round(4).T

## 2. Selection on dev vs on grouped CV, plus untuned defaults

In [ ]:
# per model, three settings: the default, the best by dev EER, the best by grouped CV
picks = {}
for m in SPACES:
    s = search[search.model == m]
    picks[(m, "default")] = DEFAULTS[m]
    picks[(m, "dev")] = json.loads(s.loc[s.dev_eer.idxmin(), "params"])
    picks[(m, "cv")] = json.loads(s.loc[s.cv_eer.idxmin(), "params"])
    print(f"{m:7s} same setting chosen by dev and CV: {picks[(m, 'dev')] == picks[(m, 'cv')]}")

# refit each pick on all of train and score train and dev (eval is not used here)
for (m, sel), params in picks.items():
    run(1, m, X, train, [("asvspoof", "train", train), ("asvspoof", "dev", dev)],
        params=params, experiment="e3", feature_set=f"lfcc40dd_{sel}sel" if sel != "default" else "lfcc40dd_default")

## 3. Gap table: train vs speaker-grouped CV vs dev (EER %)

In [ ]:
from deepvoice.models import grouped_cv_eer

r = pd.read_csv(ROOT / "results/e3.csv")
# overfitting / split-sensitivity table: train vs grouped CV vs dev EER for every pick
rows = []
for (m, sel), params in picks.items():
    fs = f"lfcc40dd_{sel}sel" if sel != "default" else "lfcc40dd_default"
    rr = r[r.config == f"e3_{m}_{fs}_s0"].set_index("split")
    # the default setting was not in the search, so its CV EER is computed here
    if sel == "default":
        cv = grouped_cv_eer(m, params, train, X)
    else:
        s = search[(search.model == m) & (search.params == json.dumps(params))]
        cv = s.cv_eer.iloc[0]
    # EERs in percent; a large train-to-dev gap means overfitting
    rows.append({"model": m, "selected by": sel, "train": 100 * rr.loc["train", "eer"], "grouped CV": 100 * cv,
                 "dev": 100 * rr.loc["dev", "eer"],
                 "dev 95% CI": f"{100*rr.loc['dev','ci_low']:.2f}-{100*rr.loc['dev','ci_high']:.2f}",
                 "fit s": rr.loc["dev", "train_time"], "params": json.dumps(params)})
gap = pd.DataFrame(rows)
gap.to_csv(ROOT / "results/e3_gap.csv", index=False, float_format="%.4g")
gap.round(3)

## 4. Learning curves (dev-selected settings)

In [ ]:
# learning curves: refit the dev-selected setting on growing shares of train
FRACS = [0.05, 0.10, 0.25, 0.50, 1.00]
for m in SPACES:
    for f in FRACS:
        # stratified subsample (same real/fake ratio), fixed seed
        sub = train if f == 1 else train.groupby("label", group_keys=False).sample(frac=f, random_state=0)
        run(1, m, X, sub, [("asvspoof", "dev", dev)], params=picks[(m, "dev")],
            experiment="e3", feature_set=f"lfcc40dd_devsel_frac{int(f * 100):03d}")
# collect the learning-curve runs: dev EER (%) by share of train and model
lc = pd.read_csv(ROOT / "results/e3.csv")
lc = lc[lc.config.str.contains("_frac")].copy()
lc["frac"] = lc.config.str.extract(r"frac(\d+)").astype(int)[0]
lc["model"] = lc.model.str.split(":").str[0]
lc.pivot_table(index="frac", columns="model", values="eer").mul(100).round(2)

## Figure

In [ ]:
COL = {"logreg": "#1baf7a", "rf": "#6250d6", "hgb": "#eda100"}
NAME = {"logreg": "logistic regression", "rf": "random forest", "hgb": "gradient boosting"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# (a) learning curves with CI band
for m in SPACES:
    d = lc[lc.model == m].sort_values("frac")
    ax[0].plot(d.frac, 100 * d.eer, marker="o", ms=6, lw=2, color=COL[m], label=NAME[m])
    ax[0].fill_between(d.frac, 100 * d.ci_low, 100 * d.ci_high, color=COL[m], alpha=0.15, lw=0)
ax[0].set_xscale("log"); ax[0].set_xticks([5, 10, 25, 50, 100], ["5%", "10%", "25%", "50%", "100%"])
ax[0].set_xlabel("share of train used"); ax[0].set_ylabel("dev EER % (band = 95% CI)")
ax[0].grid(axis="y", color=GRID, linewidth=0.6); ax[0].legend(frameon=False)
ax[0].set_title("(a) Learning curves (dev-selected settings)", loc="left", color=INK)

# (b) gap: train / grouped CV / dev for the dev-selected models
g = gap[gap["selected by"] == "dev"].set_index("model")
marks = [("train", "o", "train"), ("grouped CV", "s", "speaker-grouped CV"), ("dev", "D", "dev")]
for i, m in enumerate(SPACES):
    vals = [g.loc[m, c] for c, _, _ in marks]
    ax[1].plot([min(vals), max(vals)], [i, i], color=COL[m], lw=2)
    for (c, mk, _), v in zip(marks, vals):
        ax[1].plot(v, i, marker=mk, ms=8, color=COL[m], mec="white", mew=1.5, ls="none")
for c, mk, lab in marks:
    ax[1].plot([], [], marker=mk, ms=7, color=MUTED, ls="none", label=lab)
ax[1].set_yticks(range(len(SPACES)), [NAME[m] for m in SPACES]); ax[1].invert_yaxis()
ax[1].set_xlabel("EER %"); ax[1].grid(axis="x", color=GRID, linewidth=0.6)
ax[1].legend(frameon=False, loc="lower right")
ax[1].set_title("(b) Train vs grouped CV vs dev (tuned on dev)", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e3_tuning.png", dpi=200)
fig.savefig(ROOT / "results/figures/e3_tuning.pdf")

---
# 4. Rung 2: frozen XLS-R-300M

Layer sweep on dev, small MLP, the pre-registered selection rule, and the single eval run of the chosen model. *(Source: `notebooks/03_xlsr.ipynb`.)*

## 03 Rung 2: frozen XLS-R-300M, layer sweep (train -> dev)

Embeddings come from `deepvoice.xlsr.extract_cached` (mean-pooled every 3rd layer, fp16, cached in
`data/features/`). For each layer: standardized logistic regression (C=1, class-balanced, seed 0),
trained on train and scored on dev with a speaker-clustered 95% CI. Results: `results/rung2.csv`.
The layer for rung 2 is chosen on dev.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.models import run
from deepvoice.xlsr import LAYERS

p = pd.read_parquet("data/protocols.parquet")
# XLS-R embeddings: 1024 numbers per clip per layer (mean over time of the hidden states)
COLS = [f"x{i:04d}" for i in range(1024)]
D = {}
# cached embeddings (clips x 9 layers x 1024); the assert checks the rows
# are in protocol order, so labels line up with embeddings
for s in ["train", "dev"]:
    m = p[p.split == s][["file", "label", "speaker"]].reset_index(drop=True)
    assert Path(f"data/features/xlsr300m_{s}_files.txt").read_text().split() == list(m.file)
    D[s] = (m, np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r"))

In [ ]:
out = ROOT / "results/rung2.csv"
done = set(pd.read_csv(out).config) if out.exists() else set()
# layer sweep: one logistic regression per layer, fit on train, scored on dev
for k, layer in enumerate(LAYERS):
    if f"rung2_logreg_xlsr300m_L{layer:02d}_s0" in done:
        continue
    # table of meta columns + the 1024 embedding values of this layer
    fr = {s: pd.concat([m, pd.DataFrame(np.asarray(e[:, k], dtype=np.float32), columns=COLS)], axis=1)
          for s, (m, e) in D.items()}
    run(2, "logreg", COLS, fr["train"], [("asvspoof", "train", fr["train"]), ("asvspoof", "dev", fr["dev"])],
        params={"C": 1.0}, feature_set=f"xlsr300m_L{layer:02d}")
# dev EER (%) per layer, with its 95% CI
r = pd.read_csv(out)
r["layer"] = r.model.str.extract(r"L(\d+)").astype(int)[0]
dev = r[r.split == "dev"].sort_values("layer")
dev[["layer", "eer", "ci_low", "ci_high", "train_time"]].assign(**{c: dev[c] * 100 for c in ["eer", "ci_low", "ci_high"]}).round(3)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.fill_between(dev.layer, 100 * dev.ci_low, 100 * dev.ci_high, color="#1baf7a", alpha=0.18, lw=0)
ax.plot(dev.layer, 100 * dev.eer, marker="o", ms=6, lw=2, color="#1baf7a", label="XLS-R layer + logistic regression")
ax.axhline(0.49, color=MUTED, ls="--", lw=1)
ax.text(24.3, 0.49, "best classical\n(LFCC40 RF, untuned)", va="center", fontsize=8, color=MUTED)
ax.set_xticks(LAYERS); ax.set_xlim(-0.5, 30)
ax.set_xlabel("XLS-R layer (0 = CNN output, 24 = last)"); ax.set_ylabel("dev EER % (band = 95% CI)")
ax.grid(axis="y", color=GRID, linewidth=0.6)
ax.set_title("Frozen XLS-R-300M: dev EER by layer", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/xlsr_layers.png", dpi=200)
fig.savefig(ROOT / "results/figures/xlsr_layers.pdf")

## Small MLP on the best layers, and the rung 2 choice (decided before eval)

Rule, fixed before any eval run: take the rung 2 candidate with the lowest dev EER; every candidate
whose dev EER lies inside that model's 95% CI counts as tied, and ties are broken by speaker-grouped
5-fold CV EER on train; if CV ties too, the lower dev EER wins.

*Note (20 Nov):* an earlier version broke an exact CV tie by row order; in one re-run that picked
layer 6 and scored it on eval once (0.34% EER, kept in `results/rung2.csv` for transparency). The
final model stays layer 3, the lowest-dev-EER model as `hypotheses.md` defines it. Candidates: logistic regression on each layer, and an MLP (one hidden layer of
256, early stopping) on the tied layers.

In [ ]:
from deepvoice.models import grouped_cv_eer


# meta columns + one layer's embeddings, for the given splits
def frames(layer, splits):
    k = LAYERS.index(layer)
    return {s: pd.concat([D[s][0], pd.DataFrame(np.asarray(D[s][1][:, k], dtype=np.float32), columns=COLS)], axis=1)
            for s in splits}

r = pd.read_csv(out); dev = r[r.split == "dev"].copy()
# the lowest dev EER; any model whose dev EER is inside its 95% CI counts as tied
best = dev.loc[dev.eer.idxmin()]
tied_layers = sorted(int(m.split("L")[1]) for m in dev[dev.eer <= best.ci_high].model)
print("best:", best.model, f"{100 * best.eer:.4f}%", "| tied layers:", tied_layers)
# small neural network (one hidden layer of 256) tried on the tied layers
MLP = {"hidden_layer_sizes": (256,), "alpha": 1e-3}
for layer in tied_layers:
    if f"rung2_mlp_xlsr300m_L{layer:02d}_s0" not in set(r.config):
        fr = frames(layer, ["train", "dev"])
        run(2, "mlp", COLS, fr["train"], [("asvspoof", "train", fr["train"]), ("asvspoof", "dev", fr["dev"])],
            params=MLP, feature_set=f"xlsr300m_L{layer:02d}")
r = pd.read_csv(out); dev = r[r.split == "dev"].copy()
best = dev.loc[dev.eer.idxmin()]
# re-collect the ties (logistic regressions and MLPs) and break them by speaker-grouped CV
tied = dev[dev.eer <= best.ci_high].copy()
for i, row in tied.iterrows():
    name, layer = row.model.split(":")[0], int(row.model.split("L")[1])
    params = {"C": 1.0} if name == "logreg" else MLP
    tied.loc[i, "cv_eer"] = grouped_cv_eer(name, params, frames(layer, ["train"])["train"], COLS)
# ties on CV fall back to the lower dev EER, so the choice never depends on row order
choice = tied.sort_values(["cv_eer", "eer"]).iloc[0]
tied[["model", "eer", "ci_low", "ci_high", "cv_eer"]].assign(**{c: tied[c] * 100 for c in ["eer", "ci_low", "ci_high", "cv_eer"]}).round(4)

In [ ]:
CHOICE = choice.model
print("rung 2 model (chosen on dev + grouped CV, before eval):", CHOICE)

## Rung 2 on ASVspoof eval (scored once)

In [ ]:
from deepvoice.data import REPEATED_ATTACKS
from deepvoice.metrics import per_attack_eer

# the rung 2 choice made above, before eval was ever loaded
name, layer = CHOICE.split(":")[0], int(CHOICE.split("L")[1])
params = {"C": 1.0} if name == "logreg" else MLP
# the single eval run of the chosen model (allowed: hypotheses.md is frozen)
m = p[p.split == "eval"][["file", "label", "speaker", "attack"]].reset_index(drop=True)
assert Path("data/features/xlsr300m_eval_files.txt").read_text().split() == list(m.file)
ev = np.load("data/features/xlsr300m_eval.npy", mmap_mode="r")
D["eval"] = (m[["file", "label", "speaker"]], ev)
fr = frames(layer, ["train", "dev", "eval"])
model, rows = run(2, name, COLS, fr["train"], [("asvspoof", s, fr[s]) for s in ["train", "dev", "eval"]],
                  params=params, feature_set=f"xlsr300m_L{layer:02d}")
s = model.predict_proba(fr["eval"][COLS].to_numpy())[:, 1]
# per-attack EER: that attack's fakes against all real eval clips
pa = pd.Series(per_attack_eer(m.label, s, m.attack)).mul(100)
pa = pd.DataFrame({CHOICE: pa, "type": ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa.index]})
pa.to_csv(ROOT / "results/rung2_eval_per_attack.csv", float_format="%.2f")
print({r_["split"]: f"{100*r_['eer']:.2f}% [{100*r_['ci_low']:.2f}, {100*r_['ci_high']:.2f}]" for r_ in rows})
pa.round(2).T

---
# 5. E4: generalization

Rung 0 per attack, then all three final models refit from code and scored on dev, eval and In-the-Wild; H3 and H4 verdicts. *(Source: `notebooks/04_generalization.ipynb`.)*

## 04 Generalization: per-attack EER on ASVspoof eval

Run only after `hypotheses.md` was frozen (commit 1d896cd). Each final model is scored on eval once.
Per-attack EER = that attack's fakes against all eval real clips (`deepvoice.metrics.per_attack_eer`).
A16 and A19 reuse training algorithms A04 and A06 ("repeated"); the other 11 are unseen.

**Rung 0** (silence/duration only). The rung 0 model is the one with the lower dev EER: the
depth-3 tree (dev 28.0% vs 30.4% for logistic regression). It is refit here deterministically
(seed 0, identical to `results/rung0.csv`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import REPEATED_ATTACKS
from deepvoice.metrics import per_attack_eer
from deepvoice.models import build

p = pd.read_parquet("data/protocols.parquet")
# silence/duration features + labels for train (fit) and eval (score)
S = {s: pd.read_parquet(f"data/features/silence_{s}.parquet").merge(p[["file", "label", "speaker", "attack"]], on="file")
     for s in ["train", "eval"]}
F = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
rows = {}
# rung 0 candidates: depth-3 tree and logistic regression on 4 silence numbers
for m, params in [("tree", {"max_depth": 3}), ("logreg", {"C": 1.0})]:
    model = build(m, 0, **params).fit(S["train"][F].to_numpy(), S["train"].label.to_numpy())
    s = model.predict_proba(S["eval"][F].to_numpy())[:, 1]
    rows[m] = per_attack_eer(S["eval"].label, s, S["eval"].attack)
pa = pd.DataFrame(rows).mul(100)
# A16 and A19 repeat training attacks A04 and A06; the other 11 are unseen
pa["type"] = ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa.index]
pa.to_csv(ROOT / "results/rung0_eval_per_attack.csv", float_format="%.2f")
r0 = pd.read_csv(ROOT / "results/rung0.csv").query("split == 'eval'").set_index("model")
pa.round(1)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.2))
d = pa["tree"]
x = range(len(d))
bars = ax.bar(x, d, color=["#c3c2b7" if t == "repeated" else "#52514e" for t in pa.type], width=0.7)
for b, t in zip(bars, pa.type):
    if t == "repeated":
        b.set_hatch("////"); b.set_edgecolor("white")
for i, v in zip(x, d):
    ax.text(i, v + 1, f"{v:.0f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(50, color=MUTED, ls=":", lw=1); ax.text(-0.4, 51, "chance (50%)", fontsize=8, color=MUTED)
ax.axhline(100 * r0.loc["tree", "eer"], color="#e34948", ls="--", lw=1.2)
ax.text(-0.4, 100 * r0.loc["tree", "eer"] + 1.5, f"pooled eval EER {100 * r0.loc['tree', 'eer']:.1f}%", fontsize=8, color="#e34948")
ax.set_xticks(list(x), d.index); ax.set_ylim(0, 60)
ax.set_ylabel("eval EER % (silence/duration only)")
from matplotlib.patches import Patch

ax.legend(handles=[Patch(color="#52514e", label="unseen attack"), Patch(facecolor="#c3c2b7", hatch="////", edgecolor="white", label="repeated (A16=A04, A19=A06)")],
          frameon=False, loc="upper left", bbox_to_anchor=(0, 0.72), fontsize=8)
ax.set_title("Rung 0 on ASVspoof eval, per attack", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.png", dpi=200)
fig.savefig(ROOT / "results/figures/rung0_eval_per_attack.pdf")

## E4: the three final models on In-the-Wild (H3, H4)

The final model of each rung (chosen on dev, see notebooks 02/03 and `hypotheses.md`) is refit here
from code with its saved settings (deterministic, seed 0) and scored on train, dev, eval and
In-the-Wild (ITW). The eval numbers must equal the ones already recorded; this re-runs the same
models, it is not a new look at eval.

- **H3 (ITW part):** the rung 2 minus rung 1 margin is reported on eval and ITW; a shrinkage is
  only claimed if it is >= 10 pp.
- **H4:** on ITW, error at the dev threshold (mean of false-accept and false-reject rate) minus the
  best-threshold error (ITW EER) >= 10 pp for at least 2 of the 3 rungs.
- Rung 0 on ITW is the shortcut control: if silence alone separates ITW, ITW has a shortcut too.

In [ ]:
import json

import numpy as np

from deepvoice.features import load_xy
from deepvoice.metrics import eer, error_at_threshold
from deepvoice.models import run
from deepvoice.xlsr import LAYERS

# train for fitting; dev, eval and In-the-Wild (itw) for scoring
SPLITS = ["train", "dev", "eval", "itw"]
DATASET = {"train": "asvspoof", "dev": "asvspoof", "eval": "asvspoof", "itw": "itw"}


# meta columns + layer-`layer` XLS-R embeddings for every split
def xlsr_frames(layer):
    k, cols, out = LAYERS.index(layer), [f"x{i:04d}" for i in range(1024)], {}
    for s in SPLITS:
        meta = load_xy(s, groups=())
        assert Path(f"data/features/xlsr300m_{s}_files.txt").read_text().split() == list(meta.file)
        e = np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r")
        out[s] = pd.concat([meta, pd.DataFrame(np.asarray(e[:, k], dtype=np.float32), columns=cols)], axis=1)
    return out, cols


# rung 1 settings chosen on dev in E3 (notebook 02)
gap = pd.read_csv(ROOT / "results/e3_gap.csv")
hgb_params = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
# cached features of the three final models
lfcc = {s: load_xy(s, groups=("lfcc",), n_coef=40, deltas=2, stats=("mean", "std")) for s in SPLITS}
sil = {s: load_xy(s, groups=("silence",)) for s in SPLITS}
xl, xcols = xlsr_frames(3)
FINAL = [  # rung, model, data, feature columns, params, experiment, feature_set
    (0, "tree", sil, ["lead_sil", "trail_sil", "sil_ratio", "duration"], {"max_depth": 3}, "rung0", None),
    (1, "hgb", lfcc, list(lfcc["train"].columns[4:]), hgb_params, "rung1", "lfcc40dd_e3devsel"),
    (2, "logreg", xl, xcols, {"C": 1.0}, "rung2", "xlsr300m_L03"),
]

In [ ]:
rows, scores = [], {}
for rung, name, D, X, params, exp, fs in FINAL:
    config = f"{exp}_{name}_{fs}_s0" if fs else f"{exp}_{name}_s0"
    # the eval EER recorded earlier; refitting must reproduce it exactly
    before = pd.read_csv(ROOT / f"results/{exp}.csv").query("config == @config and split == 'eval'").eer
    model, res = run(rung, name, X, D["train"], [(DATASET[s], s, D[s]) for s in SPLITS],
                     params=params, experiment=exp, feature_set=fs)
    after = {r["split"]: r for r in res}  # results CSVs keep 6 significant digits
    assert len(before) == 1 and np.isclose(before.iloc[0], after["eval"]["eer"], rtol=1e-5, atol=0), "eval not reproduced"
    # P(fake) of every clip in every split
    sc = {s: model.predict_proba(D[s][X].to_numpy())[:, 1] for s in SPLITS}
    scores[rung] = sc
    # dev threshold: the score at which dev's false-accept and false-reject rates are equal
    thr = eer(D["dev"].label, sc["dev"])[1]
    row = {"rung": rung, "model": f"{name}:{fs}" if fs else name, "dev_threshold": thr}
    # EER = error at the best threshold for that set; error_at_threshold
    # = error when the dev threshold is used unchanged (H4)
    for s in ["dev", "eval", "itw"]:
        row[f"{s}_eer"], row[f"{s}_ci"] = after[s]["eer"], f"{100*after[s]['ci_low']:.1f}-{100*after[s]['ci_high']:.1f}"
        row[f"{s}_err_at_dev_thr"] = error_at_threshold(D[s].label, sc[s], thr)
    rows.append(row)
e4 = pd.DataFrame(rows)
e4.to_csv(ROOT / "results/e4_itw.csv", index=False, float_format="%.6g")
show = e4.copy()
for c in [c for c in e4.columns if c.endswith("_eer") or c.endswith("_thr") and c != "dev_threshold"]:
    show[c] = (100 * show[c]).round(2)
show

## Rung 1 per attack on ASVspoof eval (H2)

Same refit rung 1 model as above (gradient boosting on LFCC-40 with deltas), so its eval scores are
the recorded ones. Per-attack EER = that attack's fakes against all eval real clips. This table is
`results/rung1_eval_per_attack.csv` and Table 4 of the report.

In [ ]:
ev = lfcc["eval"]
pa1 = pd.Series(per_attack_eer(ev.label, scores[1]["eval"], ev.attack)).mul(100).rename("hgb_lfcc40").to_frame()
pa1["type"] = ["repeated" if a in REPEATED_ATTACKS else "unseen" for a in pa1.index]
pa1.to_csv(ROOT / "results/rung1_eval_per_attack.csv", float_format="%.2f")
pa1.round(2).T

## Hypothesis verdicts from E4 (rules from the frozen `hypotheses.md`)

In [ ]:
by = e4.set_index("rung")
# H1a: rung 0 eval EER < 25% true, >= 35% false, otherwise inconclusive
h1a = by.loc[0, "eval_eer"]
# H2: rung 1 worst unseen-attack EER >= 2x the mean EER of the repeated attacks (A16, A19)
worst_unseen = pa1.loc[pa1.type == "unseen", "hgb_lfcc40"]
rep_mean = pa1.loc[pa1.type == "repeated", "hgb_lfcc40"].mean()
# H3 (in-distribution): rung 2 beats rung 1 on eval by >= 5 pp
m_eval = by.loc[1, "eval_eer"] - by.loc[2, "eval_eer"]
m_itw = by.loc[1, "itw_eer"] - by.loc[2, "itw_eer"]
shrink = m_eval - m_itw
# H4: on ITW, error at the dev threshold minus ITW EER >= 10 pp for at least 2 of 3 rungs
h4_gaps = by["itw_err_at_dev_thr"] - by["itw_eer"]
verdicts = pd.DataFrame([
    {"hypothesis": "H1a",
     "evidence": f"rung 0 eval EER {100*h1a:.1f}% (true < 25%, false >= 35%)",
     "verdict": "true" if h1a < 0.25 else "false" if h1a >= 0.35 else "inconclusive"},
    {"hypothesis": "H2",
     "evidence": f"rung 1 worst unseen attack {worst_unseen.idxmax()} {worst_unseen.max():.2f}% vs repeated mean {rep_mean:.2f}% "
                 f"({worst_unseen.max() / rep_mean:.0f}x; rule >= 2x)",
     "verdict": "true" if worst_unseen.max() >= 2 * rep_mean else "false"},
    {"hypothesis": "H3 (in-distribution)",
     "evidence": f"rung 1 minus rung 2 eval EER: {100*m_eval:.1f} pp (rule >= 5 pp)",
     "verdict": "true" if m_eval >= 0.05 else "false"},
    {"hypothesis": "H3 (ITW part)",
     "evidence": f"rung2-rung1 margin: eval {100*m_eval:.1f} pp, ITW {100*m_itw:.1f} pp, shrink {100*shrink:.1f} pp",
     "verdict": "edge shrinks (>= 10 pp)" if shrink >= 0.10 else "no shrinkage of >= 10 pp (not decidable as shrinking)"},
    {"hypothesis": "H4",
     "evidence": "ITW error at dev threshold minus ITW EER: " + ", ".join(f"rung {r}: {100*g:.1f} pp" for r, g in h4_gaps.items()),
     "verdict": "true" if (h4_gaps >= 0.10).sum() >= 2 else "false"},
])
verdicts.to_csv(ROOT / "results/e4_hypotheses.csv", index=False)
verdicts

**Reading H4.** The frozen rule compares the error at the dev threshold with the best-threshold
error, which `hypotheses.md` defines as the EER. Rung 1's In-the-Wild EER is 77.6%, worse than the 50%
of guessing: its scores are *inverted* on ITW (it ranks real clips as more fake). Its error at the dev
threshold is about 50%, so its gap is negative (-27 pp) and it cannot count toward H4. Only rung 0
reaches 10 pp, so H4 is false by its rule. The plain-language finding is the opposite of "thresholds
survive": every rung is near chance at its dev threshold on ITW. The rung is so far off that a better
threshold cannot rescue it, which the pre-registered gap does not measure. H1b is decided in notebook 05.

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(6.4, 3.4))
names = ["rung 0\nsilence/duration", "rung 1\nLFCC + gradient boosting", "rung 2\nfrozen XLS-R + LR"]
bars = [("eval_eer", "ASVspoof eval (EER)", "#52514e", None),
        ("itw_eer", "In-the-Wild (EER)", "#e34948", None),
        ("itw_err_at_dev_thr", "In-the-Wild at dev threshold", "#f2a3a2", "////")]
w = 0.26
for j, (col, lab, colr, hatch) in enumerate(bars):
    x = np.arange(3) + (j - 1) * w
    b = ax.bar(x, 100 * e4[col], width=w - 0.02, color=colr, hatch=hatch, edgecolor="white", label=lab)
    for xi, v in zip(x, 100 * e4[col]):
        ax.text(xi, v + 1, f"{v:.0f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(50, color=MUTED, ls=":", lw=1); ax.text(-0.45, 51, "chance (50%)", fontsize=8, color=MUTED)
ax.set_xticks(range(3), names); ax.set_ylim(0, 90); ax.set_ylabel("error %")
ax.grid(axis="y", color=GRID, linewidth=0.6); ax.legend(frameon=False, fontsize=8, loc="upper right")
ax.set_title("Same models, new data source", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e4_itw.png", dpi=200)
fig.savefig(ROOT / "results/figures/e4_itw.pdf")

---
# 6. E5: shortcuts, mitigation, calibration, failure analysis

Silence trimming and gain tests (H1b), augmentation retraining, ECE and temperature scaling, failures by duration, silence, gender and speaker. *(Source: `notebooks/05_shortcuts_calibration.ipynb`.)*

## 05 E5: shortcuts, mitigation (and later calibration)

All models are the final ones (chosen on dev, notebooks 02-03), refit from code with seed 0.
Test-time changes (`deepvoice.features.apply_transform`): **trim** (cut leading/trailing silence
with the group-S VAD), **gain-12** (12 dB quieter), **gain+6** (6 dB louder, clipped). Features of
changed audio are cached as `<group>_<split>_<change>.parquet`.

- **H1b** (frozen): trimming silence raises the rung 1 eval EER by >= 5 pp with the paired
  speaker-bootstrap CI excluding 0.
- XLS-R normalizes every clip to zero mean and unit variance, so gain changes cannot change its
  input (apart from clipped samples at +6 dB); it gets the trim test only.
- **Mitigation (E5.2):** rung 1 retrained on augmented train audio (trim, random 0-1 s noise padding,
  random -12..+6 dB gain; seeded per clip), same hyperparameters, compared on the same clips.

Results: `results/e5.csv` (standard columns; `split` names the change) and
`results/e5_diffs.csv` (paired differences with 95% CIs).

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import load_itw
from deepvoice.features import feature_tables, load_xy
from deepvoice.metrics import paired_speaker_bootstrap_diff
from deepvoice.models import run

# cached features of changed audio (computed once; loaded here)
p = pd.read_parquet("data/protocols.parquet")
sets = {"eval": p[p.split == "eval"].reset_index(drop=True), "itw": load_itw().reset_index(drop=True)}
for change in ["trim", "gain-12", "gain+6"]:
    for s, df in sets.items():
        feature_tables(df, f"{s}_{change}", transform=change)
feature_tables(p[p.split == "train"].reset_index(drop=True), "train_augment", transform="augment")

# rung 1 features (LFCC-40 + deltas) and its dev-selected gradient-boosting settings
FS = {"groups": ("lfcc",), "n_coef": 40, "deltas": 2, "stats": ("mean", "std")}
gap = pd.read_csv(ROOT / "results/e3_gap.csv")
HGB = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
# score sets: dev, eval, itw, and the same clips after each test-time change
# (trim = leading/trailing silence cut; gain-12 / gain+6 = 12 dB quieter / 6 dB louder)
EV = {"dev": load_xy("dev", **FS)}
for s in ["eval", "itw"]:
    EV[s] = load_xy(s, **FS)
    for change in ["trim", "gain-12", "gain+6"]:
        EV[f"{s}_{change}"] = load_xy(s, variant=change, **FS)
# original train and augmented train (silence trimmed, noise padded, random gain)
train, train_aug = load_xy("train", **FS), load_xy("train", variant="augment", **FS)
X = list(train.columns[4:])
DATASET = lambda s: "itw" if s.startswith("itw") else "asvspoof"
print({k: len(v) for k, v in EV.items()})

## Rung 1: original vs augmented training, on clean and changed audio

In [ ]:
models, scores = {}, {}
# same model and settings fit twice: on original and on augmented train
for tag, tr in [("lfcc40dd_e3devsel", train), ("lfcc40dd_e3devsel_augment", train_aug)]:
    m, _ = run(1, "hgb", X, tr, [(DATASET(s), s, EV[s]) for s in EV], params=HGB,
               experiment="e5", feature_set=tag)
    models[tag] = m
    # P(fake) on every score set, kept for the paired comparisons below
    scores[tag] = {s: m.predict_proba(EV[s][X].to_numpy())[:, 1] for s in EV}
e5 = pd.read_csv(ROOT / "results/e5.csv")
e5[e5.rung == 1].assign(eer=lambda d: (100 * d.eer).round(2)).pivot_table(index="split", columns="model", values="eer")

## Paired differences (EER of B minus EER of A on the same clips, 95% CI)

In [ ]:
rows = []
orig, aug = scores["lfcc40dd_e3devsel"], scores["lfcc40dd_e3devsel_augment"]
# paired speaker bootstrap: resample speakers, recompute both EERs on
# the same resample, report the difference and its 95% CI
for s in ["eval", "itw"]:
    for change in ["trim", "gain-12", "gain+6"]:  # same clips and labels, changed audio
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s], orig[f"{s}_{change}"], EV[s].speaker)
        rows.append({"model": "rung 1 (original)", "A": f"{s} clean", "B": f"{s} {change}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
    # does augmentation help? augmented minus original, on clean and trimmed audio
    for v in ["", "_trim"]:
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s + v], aug[s + v], EV[s].speaker)
        rows.append({"model": "augmented minus original", "A": f"original on {s}{v.replace('_', ' ')}",
                     "B": f"augmented on {s}{v.replace('_', ' ')}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
diffs = pd.DataFrame(rows)
diffs.to_csv(ROOT / "results/e5_diffs.csv", index=False, float_format="%.3f")
diffs.round(2)

## H1b verdict (rule from the frozen `hypotheses.md`)

In [ ]:
h = diffs[(diffs.model == "rung 1 (original)") & (diffs.B == "eval trim")].iloc[0]
h1b = h.diff_pp >= 5 and h.ci_low_pp > 0
print(f"Trimming raises rung 1 eval EER by {h.diff_pp:.1f} pp [{h.ci_low_pp:.1f}, {h.ci_high_pp:.1f}] -> H1b {'TRUE' if h1b else 'FALSE'}")

## Rung 2 (frozen XLS-R layer 3 + logistic regression): clean vs trimmed

In [ ]:
from deepvoice.xlsr import LAYERS, extract_cached

extract_cached(sets["eval"], "eval_trim", transform="trim")   # cached: loads, never recomputes
extract_cached(sets["itw"], "itw_trim", transform="trim")
# rung 2 = layer 3 of XLS-R
K, COLS = LAYERS.index(3), [f"x{i:04d}" for i in range(1024)]
META = {"train": train, "dev": EV["dev"], "eval": EV["eval"], "eval_trim": EV["eval"], "itw": EV["itw"], "itw_trim": EV["itw"]}


# meta + layer-3 embeddings of a (possibly trimmed) score set, rows checked against protocol
def xframe(name):
    meta = META[name]
    assert Path(f"data/features/xlsr300m_{name}_files.txt").read_text().split() == list(meta.file)
    e = np.asarray(np.load(f"data/features/xlsr300m_{name}.npy", mmap_mode="r")[:, K], dtype=np.float32)
    return pd.concat([meta[["file", "label", "speaker"]], pd.DataFrame(e, columns=COLS)], axis=1)


R2 = {s: xframe(s) for s in META}
# refit rung 2 and score every set, including the trimmed audio
m2, _ = run(2, "logreg", COLS, R2["train"], [(DATASET(s), s, R2[s]) for s in META if s != "train"],
            params={"C": 1.0}, experiment="e5", feature_set="xlsr300m_L03")
s2 = {s: m2.predict_proba(R2[s][COLS].to_numpy())[:, 1] for s in META if s != "train"}
# H1b-style test for rung 2: EER change when silence is trimmed
for s in ["eval", "itw"]:
    d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, s2[s], s2[f"{s}_trim"], EV[s].speaker)
    rows.append({"model": "rung 2 (XLS-R)", "A": f"{s} clean", "B": f"{s} trim", "clips": s,
                 "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
diffs = pd.DataFrame(rows)
diffs.to_csv(ROOT / "results/e5_diffs.csv", index=False, float_format="%.3f")
diffs[diffs.model == "rung 2 (XLS-R)"].round(2)

## Calibration (E5.3): reliability, ECE (15 bins), temperature scaling fit on dev

All three final models were trained with balanced class weights, so their probabilities assume a
50/50 world while the data is about 90% fake (ITW: 37%); temperature scaling (one number, no bias)
can soften over-confidence but cannot fix that prior shift.

In [ ]:
from deepvoice.metrics import apply_temperature, ece, fit_temperature, reliability
from deepvoice.models import build

# refit rung 0 so all three rungs' probabilities are available
SIL = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
silf = {s: load_xy(s, groups=("silence",)) for s in ["train", "dev", "eval", "itw"]}
m0 = build("tree", 0, max_depth=3).fit(silf["train"][SIL].to_numpy(), silf["train"].label.to_numpy())
PROBS = {0: {s: m0.predict_proba(silf[s][SIL].to_numpy())[:, 1] for s in ["dev", "eval", "itw"]},
         1: {s: orig[s] for s in ["dev", "eval", "itw"]},
         2: {s: s2[s] for s in ["dev", "eval", "itw"]}}
cal = []
# temperature scaling: one number t fitted on dev that rescales the model's confidence;
# ECE (expected calibration error) = average gap between predicted P(fake) and the
# observed fake rate, before and after scaling
for rung, pr in PROBS.items():
    t = fit_temperature(EV["dev"].label, pr["dev"])
    for s in ["dev", "eval", "itw"]:
        y = EV[s].label
        cal.append({"rung": rung, "split": s, "temperature": t, "ece_before": ece(y, pr[s]),
                    "ece_after": ece(y, apply_temperature(pr[s], t))})
cal = pd.DataFrame(cal)
cal.to_csv(ROOT / "results/e5_calibration.csv", index=False, float_format="%.4f")
cal.assign(ece_before=lambda d: (100 * d.ece_before).round(1), ece_after=lambda d: (100 * d.ece_after).round(1))

## Failure analysis (E5.5): where do rungs 1 and 2 fail?

In [ ]:
from deepvoice.data import load_gender
from deepvoice.metrics import eer

# speaker -> sex, from the ASV enrolment lists (not every speaker is listed)
gender = load_gender()
fail = []
# EER inside each bucket of a factor (e.g. clip duration); buckets with one class are skipped
def bucket_eers(rung, s, sc, factor, buckets):
    d = EV[s][["label"]].assign(score=sc, b=buckets)
    for b, g in d.groupby("b", observed=True):
        if g.label.nunique() == 2:
            fail.append({"rung": rung, "dataset": s, "factor": factor, "bucket": str(b),
                         "n": len(g), "eer": eer(g.label, g.score)[0]})

# where do rungs 1 and 2 fail? by duration, by silence ratio, and by gender (eval)
for rung, sc in [(1, orig), (2, s2)]:
    for s in ["eval", "itw"]:
        sf = silf[s]
        bucket_eers(rung, s, sc[s], "duration (s)", pd.cut(sf.duration, [0, 2, 3, 4, 6, 100]))
        bucket_eers(rung, s, sc[s], "silence ratio", pd.cut(sf.sil_ratio, [-0.01, 0.2, 0.35, 0.5, 1]))
    bucket_eers(rung, "eval", sc["eval"], "gender", EV["eval"].speaker.map(gender).fillna("unknown"))
fail = pd.DataFrame(fail)
fail.to_csv(ROOT / "results/e5_failures.csv", index=False, float_format="%.4f")
fail.assign(eer=lambda d: (100 * d.eer).round(1)).pivot_table(index=["dataset", "factor", "bucket"], columns="rung", values="eer")

In [ ]:
# per In-the-Wild speaker (rung 2): EER for each of the 54 speakers
spk = []
for name, g in EV["itw"].assign(score=s2["itw"]).groupby("speaker"):
    spk.append({"speaker": name, "n": len(g), "fake_share": g.label.mean(), "eer": eer(g.label, g.score)[0]})
spk = pd.DataFrame(spk).sort_values("eer")
spk.to_csv(ROOT / "results/e5_itw_speakers.csv", index=False, float_format="%.4f")
print("speakers with EER < 20%:", (spk.eer < 0.2).sum(), "| above 50% (worse than chance):", (spk.eer > 0.5).sum(), "of", len(spk))
pd.concat([spk.head(5), spk.tail(5)]).round(3)

In [ ]:
# the most confident mistakes of rung 2 on eval, at its dev threshold
thr2 = eer(EV["dev"].label, s2["dev"])[1]
ev = EV["eval"][["file", "label", "attack", "speaker"]].assign(score=s2["eval"], duration=silf["eval"].duration,
                                                                 lead_sil=silf["eval"].lead_sil)
fa = ev[(ev.label == 1) & (ev.score < thr2)].sort_values("score").head(200)   # fakes let through
fr = ev[(ev.label == 0) & (ev.score >= thr2)].sort_values("score", ascending=False).head(200)  # real flagged
print(f"dev threshold {thr2:.3f}: {len(ev[(ev.label == 1) & (ev.score < thr2)])} fakes accepted, {len(ev[(ev.label == 0) & (ev.score >= thr2)])} real clips rejected")
print("most confident false accepts, by attack:", fa.attack.value_counts().head(6).to_dict())
print("median duration / lead silence: false accepts", fa.duration.median().round(2), fa.lead_sil.median().round(2),
      "| false rejects", fr.duration.median().round(2), fr.lead_sil.median().round(2),
      "| all eval", ev.duration.median().round(2), ev.lead_sil.median().round(2))
print("false rejects: speakers", fr.speaker.nunique(), "| top:", fr.speaker.value_counts().head(5).to_dict())

## Figures

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
e5 = pd.read_csv(ROOT / "results/e5.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
conds = ["eval", "eval_trim", "eval_gain-12", "eval_gain+6", "itw", "itw_trim"]
labels = ["eval", "eval\ntrimmed", "eval\n-12 dB", "eval\n+6 dB", "ITW", "ITW\ntrimmed"]
series = [("hgb:lfcc40dd_e3devsel", "rung 1 (LFCC)", "#eda100", None),
          ("hgb:lfcc40dd_e3devsel_augment", "rung 1 + augmentation", "#eda100", "////"),
          ("logreg:xlsr300m_L03", "rung 2 (XLS-R)", "#1baf7a", None)]
w = 0.27
for j, (mname, lab, col, hatch) in enumerate(series):
    d = e5[e5.model == mname].set_index("split")
    x = np.arange(len(conds)) + (j - 1) * w
    vals = [100 * d.eer.get(c, np.nan) for c in conds]
    ax[0].bar(x, vals, width=w - 0.02, color=col, hatch=hatch, edgecolor="white", label=lab)
    for xi, v in zip(x, vals):
        if not np.isnan(v):
            ax[0].text(xi, v + 1, f"{v:.0f}", ha="center", fontsize=7, color=INK)
ax[0].axhline(50, color=MUTED, ls=":", lw=1)
ax[0].set_xticks(range(len(conds)), labels); ax[0].set_ylim(0, 95); ax[0].set_ylabel("EER %")
ax[0].grid(axis="y", color=GRID, linewidth=0.6); ax[0].legend(frameon=False, fontsize=8, loc="upper left")
ax[0].set_title("(a) Test-time changes and augmentation (XLS-R is gain-invariant: no gain bars)", loc="left", color=INK)

for s, col, lab in [("dev", "#c3c2b7", "ASVspoof dev"), ("eval", "#52514e", "ASVspoof eval"), ("itw", "#e34948", "In-the-Wild")]:
    r = reliability(EV[s].label, PROBS[2][s])
    e = cal[(cal.rung == 2) & (cal.split == s)].ece_before.iloc[0]
    ax[1].plot(r.mean_prob, r.frac_fake, marker="o", ms=4, lw=1.8, color=col, label=f"{lab} (ECE {100 * e:.0f}%)")
ax[1].plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1)
ax[1].set_xlabel("predicted P(fake)"); ax[1].set_ylabel("observed fake rate")
ax[1].grid(color=GRID, linewidth=0.6); ax[1].legend(frameon=False, fontsize=8, loc="lower right")
ax[1].set_title("(b) Reliability of rung 2 (before scaling)", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/e5_shortcuts_calibration.png", dpi=200)
fig.savefig(ROOT / "results/figures/e5_shortcuts_calibration.pdf")

---
# 7. E1 cost table and the final model

Extraction, training and scoring cost per rung; saving and checking the final model. *(Source: `notebooks/06_cost_and_final_model.ipynb`.)*

## 06 E1 cost table and the final model

**Cost per rung**, measured the same way for all three final models: feature extraction per clip on
200 random eval clips (silence and classical features on one CPU core; XLS-R on the Apple M5 GPU,
`mps`), training time and scoring latency from the recorded runs (`results/rung*.csv`), and model
size (serialized bytes, trainable and frozen parameters). Output: `results/cost_table.csv`.

**Final model** = rung 2 (frozen XLS-R-300M layer 3 + logistic regression), the best model on ASVspoof
eval. Saved to `model/` with its metadata (layer, dev EER threshold, results, sha256); scored with
`python -m deepvoice.predict`. See `model/MODEL_CARD.md`.

In [ ]:
import hashlib
import io
import json
import os
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice import predict, xlsr
from deepvoice.features import clip_features, load_audio, load_xy
from deepvoice.metrics import eer
from deepvoice.models import build
from deepvoice.silence import silence_features

p = pd.read_parquet("data/protocols.parquet")
# 200 random eval clips used only for timing feature extraction (no scores computed)
sample = list(p[p.split == "eval"].sample(200, random_state=0).path)

## Feature extraction time per clip

In [ ]:
def per_clip_ms(fn):
    fn(sample[0])  # warm-up
    t0 = time.perf_counter()
    for path in sample:
        fn(path)
    return (time.perf_counter() - t0) / len(sample) * 1000

ext = {0: per_clip_ms(lambda q: silence_features(load_audio(q))),
       1: per_clip_ms(clip_features)}  # computes MFCC + LFCC + spectral: an upper bound for LFCC alone
xm = xlsr.load_model()
ext[2] = xlsr.time_it(sample, xm)["s_per_clip"] * 1000
frozen_params = sum(t.numel() for t in xm.parameters())
{k: round(v, 1) for k, v in ext.items()}

## The three final models: refit (deterministic, seed 0), size, recorded train and scoring time

In [ ]:
# refit the three final models on train to measure size and parameter counts
gap = pd.read_csv("results/e3_gap.csv")
hgb_params = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
SIL = ["lead_sil", "trail_sil", "sil_ratio", "duration"]
sil = {s: load_xy(s, groups=("silence",)) for s in ["train", "dev"]}
lf = {s: load_xy(s, groups=("lfcc",), n_coef=40) for s in ["train", "dev"]}
LF = list(lf["train"].columns[4:])
K = xlsr.LAYERS.index(3)
xl = {s: np.asarray(np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r")[:, K], dtype=np.float32) for s in ["train", "dev"]}
y = {s: sil[s].label.to_numpy() for s in ["train", "dev"]}

m0 = build("tree", 0, max_depth=3).fit(sil["train"][SIL].to_numpy(), y["train"])
m1 = build("hgb", 0, **hgb_params).fit(lf["train"][LF].to_numpy(), y["train"])
m2 = build("logreg", 0, C=1.0).fit(xl["train"], y["train"])

# size of the saved model file, in bytes
def nbytes(m):
    b = io.BytesIO(); joblib.dump(m, b); return b.tell()

# what each model learns: tree nodes, boosted-tree nodes, or LR weights + bias
trainable = {0: int(m0.tree_.node_count),
             1: int(sum(len(pr[0].nodes) for pr in m1._predictors)),
             2: int(m2[-1].coef_.size + m2[-1].intercept_.size)}
# where each final model's recorded results live
CONFIG = {0: ("rung0", "rung0_tree_s0"), 1: ("rung1", "rung1_hgb_lfcc40dd_e3devsel_s0"), 2: ("rung2", "rung2_logreg_xlsr300m_L03_s0")}
# cost table (E1): extraction time, training time, scoring time, size, params, EER
rows = []
for rung, (exp, cfg) in CONFIG.items():
    r = pd.read_csv(f"results/{exp}.csv").query("config == @cfg").set_index("split")
    rows.append({
        "rung": rung,
        "model": {0: "depth-3 tree, 4 silence/duration numbers", 1: "gradient boosting, LFCC-40 + deltas (240)",
                  2: "frozen XLS-R-300M layer 3 + logistic regression"}[rung],
        "extract_ms_per_clip": ext[rung], "extract_device": "mps (GPU)" if rung == 2 else "1 CPU core",
        "train_s": r.loc["dev", "train_time"], "score_ms_per_clip": r.loc["eval", "latency"],
        "model_kb": nbytes([m0, m1, m2][rung]) / 1024, "trainable_params_or_nodes": trainable[rung],
        "frozen_params": frozen_params if rung == 2 else 0,
        "eval_eer_pct": 100 * r.loc["eval", "eer"], "itw_eer_pct": 100 * r.loc["itw", "eer"]})
cost = pd.DataFrame(rows)
cost.to_csv("results/cost_table.csv", index=False, float_format="%.4g")
cost.round(3)

## Save the final model (rung 2) and check the scorer reproduces the pipeline

In [ ]:
# the decision threshold shipped with the model: the dev EER threshold
s_dev = m2.predict_proba(xl["dev"])[:, 1]
dev_eer, thr = eer(y["dev"], s_dev)
Path("model").mkdir(exist_ok=True)
# save the classifier and its sha256 so a download can be verified
joblib.dump(m2, Path("model") / predict.MODEL_FILE)
sha = hashlib.sha256((Path("model") / predict.MODEL_FILE).read_bytes()).hexdigest()
r2 = pd.read_csv("results/rung2.csv").query("config == 'rung2_logreg_xlsr300m_L03_s0'").set_index("split")
# model card metadata: what it is, what it was trained on, threshold and EERs
meta = {"name": "DeepVoice rung 2", "xlsr_model": xlsr.MODEL, "layer": 3, "pooling": "mean over time",
        "classifier": "StandardScaler + LogisticRegression(C=1, class_weight=balanced)",
        "train_data": "ASVspoof 2019 LA train (25,380 clips, 16 kHz mono)", "seed": 0,
        "threshold": thr, "threshold_rule": "EER threshold on ASVspoof 2019 LA dev",
        "eer": {s: round(float(r2.loc[s, "eer"]), 6) for s in ["dev", "eval", "itw"]},
        "eer_ci95": {s: [round(float(r2.loc[s, "ci_low"]), 6), round(float(r2.loc[s, "ci_high"]), 6)] for s in ["eval", "itw"]},
        "sha256": sha}
(Path("model") / predict.MODEL_FILE.replace(".joblib", ".json")).write_text(json.dumps(meta, indent=2) + "\n")

# the CLI path (load file -> XLS-R -> layer 3 -> classifier) must give the pipeline's scores
check = p[p.split == "dev"].reset_index(drop=True).sample(8, random_state=1)
clf, meta_loaded = predict.load()
got = predict.score(check.path.tolist(), clf, meta_loaded, xm)
want = m2.predict_proba(xl["dev"][check.index])[:, 1]
print("max |difference| CLI vs pipeline:", float(np.max(np.abs(got - want))))
assert np.allclose(got, want, atol=0.02)
meta

---
# Appendix: source code of `src/deepvoice/`

The cells above import these modules. They are reproduced here verbatim (generated from the
repository by `scripts/build_final_notebook.py`), so this notebook contains all the code used:
data loading and cleaning (`data.py`), the shared 16 kHz mono audio loader and feature extraction
(`features.py`, `silence.py`), the frozen XLS-R embeddings (`xlsr.py`), metrics with bootstrap CIs
and calibration (`metrics.py`), models, tuning and model selection (`models.py`), the cache pipeline
(`pipeline.py`) and the final-model scorer (`predict.py`).

### `src/deepvoice/__init__.py`

```python
"""DeepVoice: audited fake-voice detection study (CMPS 261)."""

import os

# Let PyTorch run ops the Apple GPU (mps) lacks on the CPU; must be set before torch is imported.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
```

### `src/deepvoice/data.py`

```python
"""ASVspoof 2019 LA protocol loading. Label 1 = fake (spoof), 0 = real (bonafide)."""

from pathlib import Path

import pandas as pd

COLUMNS = ["speaker", "utt", "env", "attack", "key"]

# Names checked against the real LA.zip contents (2026-11-05). Always iterate over these protocol
# files, never glob the flac folders: dev/eval folders hold extra files (ASV enrollment audio)
# that are not in the CM protocols (eval: 71,933 files vs 71,237 protocol rows).
SPLITS = {
    "train": ("ASVspoof2019_LA_train", "ASVspoof2019.LA.cm.train.trn.txt"),
    "dev": ("ASVspoof2019_LA_dev", "ASVspoof2019.LA.cm.dev.trl.txt"),
    "eval": ("ASVspoof2019_LA_eval", "ASVspoof2019.LA.cm.eval.trl.txt"),
}


# Eval attacks that reuse a training attack's algorithm (Wang et al. 2020, arXiv:1911.01601,
# Sec. 3.1: "A16 and A19 are the known reference systems using the same algorithms as A04 and
# A06"). They were rebuilt on different data, so they are "repeated", not identical systems.
# The other 11 eval attacks (A07-A15, A17, A18) are unseen.
REPEATED_ATTACKS = {"A16": "A04", "A19": "A06"}


def read_protocol(path):
    """Parse one protocol file: `speaker utt env attack key`, key in {bonafide, spoof}."""
    df = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMNS)
    bad = set(df["key"]) - {"bonafide", "spoof"}
    if bad:
        raise ValueError(f"unexpected key values in {path}: {bad}")
    df["label"] = (df["key"] == "spoof").astype(int)
    df["attack"] = df["attack"].replace("-", "bonafide")
    return df


def load_split(root, split):
    """Protocol dataframe for train/dev/eval with an added `path` column to each .flac."""
    folder, proto = SPLITS[split]
    root = Path(root)
    df = read_protocol(root / "ASVspoof2019_LA_cm_protocols" / proto)
    df["path"] = [str(root / folder / "flac" / f"{u}.flac") for u in df["utt"]]
    df["split"] = split
    return df


def load_protocols(root, splits=("train", "dev", "eval")):
    """One table for all splits: file, speaker, attack, label, split, path.

    Reads protocol text only (no audio), so listing eval here does not count as using it.
    """
    df = pd.concat([load_split(root, s) for s in splits], ignore_index=True)
    df = df.rename(columns={"utt": "file"})
    return df[["file", "speaker", "attack", "label", "split", "path"]]


def load_itw(root="data/release_in_the_wild"):
    """In-the-Wild metadata in the ASVspoof table format: file, speaker, attack, label, split, path.

    meta.csv has columns file, speaker, label with labels "bona-fide" / "spoof". There are no
    attack IDs (unknown generators), so attack is "bonafide" or "unknown"; split is "itw".
    """
    root = Path(root)
    m = pd.read_csv(root / "meta.csv")
    bad = set(m["label"]) - {"bona-fide", "spoof"}
    if bad:
        raise ValueError(f"unexpected labels in {root / 'meta.csv'}: {bad}")
    df = pd.DataFrame({
        "file": m["file"].str.removesuffix(".wav"),
        "speaker": m["speaker"],
        "attack": m["label"].map({"bona-fide": "bonafide", "spoof": "unknown"}),
        "label": (m["label"] == "spoof").astype(int),
        "split": "itw",
        "path": [str(root / f) for f in m["file"]],
    })
    return df


def summarize(df):
    """Per split: real/fake counts, total, number of speakers and the attack IDs present."""
    rows = []
    for split, g in df.groupby("split", sort=False):
        attacks = sorted(set(g["attack"]) - {"bonafide"})
        rows.append(
            {
                "split": split,
                "real": int((g["label"] == 0).sum()),
                "fake": int((g["label"] == 1).sum()),
                "total": len(g),
                "speakers": g["speaker"].nunique(),
                "attacks": f"{attacks[0]}-{attacks[-1]} ({len(attacks)})" if attacks else "",
            }
        )
    return pd.DataFrame(rows)


def load_gender(root="data/LA"):
    """{speaker: "female"/"male"} from the ASV enrollment protocols (<split>.<sex>.trn.txt).
    The CM protocols carry no gender; speakers without ASV enrollment are absent."""
    out = {}
    for f in sorted(Path(root, "ASVspoof2019_LA_asv_protocols").glob("*male.trn.txt")):
        sex = "female" if ".female." in f.name else "male"
        for line in f.read_text().splitlines():
            if line.strip():
                out[line.split()[0]] = sex
    return out
```

### `src/deepvoice/features.py`

```python
"""Audio loading and features: one loader for every dataset (16 kHz mono), MFCC/LFCC/spectral
features pooled over time, cached to data/ and never recomputed.

We compute the largest setting once (40 coefficients, static + delta + delta-delta, 7 pooling
statistics). Smaller settings for the E2 ablations (13/20 coefficients, no deltas, mean+std only)
are column subsets of the same numbers: the first k DCT coefficients do not depend on how many
are kept.
"""

import time
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from joblib import Parallel, delayed
from scipy.fft import dct

SR = 16000
N_FFT, WIN, HOP = 512, 400, 160  # 32 ms FFT, 25 ms window, 10 ms hop
N_CEP = 40
N_MELS, N_LIN = 80, 70
STATS = ["mean", "std", "min", "max", "p10", "p50", "p90"]
SPECTRAL = ["centroid", "bandwidth", "rolloff", "flatness", "zcr", "hf_ratio"]
GROUPS = ["mfcc", "lfcc", "spec"]


def load_audio(path, sr=SR):
    """The single loader for ASVspoof and In-the-Wild: float32 mono at `sr` Hz."""
    y, orig_sr = sf.read(path, dtype="float32", always_2d=True)
    y = y.mean(axis=1)
    if orig_sr != sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)
    return y.astype(np.float32)


def _linear_filterbank(n_filters=N_LIN, n_fft=N_FFT, sr=SR):
    """Triangular filters evenly spaced on a linear Hz scale (the LFCC counterpart of mel)."""
    freqs = np.linspace(0, sr / 2, n_fft // 2 + 1)
    edges = np.linspace(0, sr / 2, n_filters + 2)
    lo, mid, hi = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    return np.maximum(0, np.minimum((freqs - lo) / (mid - lo), (hi - freqs) / (hi - mid)))


MEL_FB = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS)
LIN_FB = _linear_filterbank()


def _cepstra(power, fb):
    """Log filterbank energies -> DCT -> first N_CEP coefficients, plus delta and delta-delta."""
    c = dct(10 * np.log10(fb @ power + 1e-10), type=2, norm="ortho", axis=0)[:N_CEP]
    d1 = librosa.feature.delta(c, order=1, mode="nearest")
    d2 = librosa.feature.delta(c, order=2, mode="nearest")
    return np.concatenate([c, d1, d2])  # (3 * N_CEP, frames)


def _pool(x):
    """(features, frames) -> (features * len(STATS),), statistic-major per feature."""
    p10, p50, p90 = np.percentile(x, [10, 50, 90], axis=1)
    return np.stack([x.mean(1), x.std(1), x.min(1), x.max(1), p10, p50, p90], axis=1).ravel()


def frame_features(y, sr=SR):
    """Frame-level MFCC, LFCC (each 40 + deltas) and spectral descriptors of one clip."""
    if len(y) < WIN:
        y = np.pad(y, (0, WIN - len(y)))
    power = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP, win_length=WIN)) ** 2
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    total = power.sum(axis=0) + 1e-10
    spec = np.stack([
        librosa.feature.spectral_centroid(S=power, sr=sr, n_fft=N_FFT)[0],
        librosa.feature.spectral_bandwidth(S=power, sr=sr, n_fft=N_FFT)[0],
        librosa.feature.spectral_rolloff(S=power, sr=sr, n_fft=N_FFT, roll_percent=0.85)[0],
        librosa.feature.spectral_flatness(S=power)[0],
        librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP)[0][: power.shape[1]],
        power[freqs >= 4000].sum(axis=0) / total,  # high-frequency energy ratio (>= 4 kHz)
    ])
    return {"mfcc": _cepstra(power, MEL_FB), "lfcc": _cepstra(power, LIN_FB), "spec": spec}


def column_names(group):
    if group == "spec":
        rows = SPECTRAL
    else:
        rows = [f"{group}_d{d}_c{c:02d}" for d in range(3) for c in range(N_CEP)]
    return [f"{r}_{s}" if group != "spec" else f"spec_{r}_{s}" for r in rows for s in STATS]


def apply_transform(y, transform=None, key=None):
    """Audio changes for E5. Test-time: None, "trim" (cut leading/trailing silence with the
    group-S VAD), "gain-12" (12 dB quieter), "gain+6" (6 dB louder, clipped to +-1).
    Training-time "augment": trim, then pad each end with 0-1 s of background noise at -80 to
    -50 dBFS, then a random gain of -12 to +6 dB (clipped). Its randomness is seeded by `key`
    (the clip id), so the augmented set is reproducible."""
    if transform is None:
        return y
    if transform == "augment":
        import zlib

        from deepvoice.silence import trim

        rng = np.random.default_rng(zlib.crc32(str(key).encode()))
        y = trim(y)
        pads = [rng.normal(0, 10 ** (rng.uniform(-80, -50) / 20), int(rng.uniform(0, 1) * SR))
                for _ in range(2)]
        y = np.concatenate([pads[0], y, pads[1]]) * 10 ** (rng.uniform(-12, 6) / 20)
        return np.clip(y, -1, 1).astype(np.float32)
    if transform == "trim":
        from deepvoice.silence import trim

        return trim(y)
    if transform == "gain-12":
        return (y * 10 ** (-12 / 20)).astype(np.float32)
    if transform == "gain+6":
        return np.clip(y * 10 ** (6 / 20), -1, 1).astype(np.float32)
    raise ValueError(f"unknown transform {transform!r}")


def clip_features(path, transform=None):
    """Pooled feature vectors {group: 1-D float32 array} for one audio file."""
    f = frame_features(apply_transform(load_audio(path), transform, key=Path(path).stem))
    return {g: _pool(f[g]).astype(np.float32) for g in GROUPS}


def feature_tables(df, name, root="data/features", n_jobs=-1, transform=None):
    """{group: DataFrame(file + pooled columns)} for every row of `df`, cached as
    <root>/<group>_<name>.parquet. Existing caches are loaded, never recomputed. `transform`
    (see apply_transform) changes the audio first; give such a cache its own `name`."""
    root = Path(root)
    paths = {g: root / f"{g}_{name}.parquet" for g in GROUPS}
    if all(p.exists() for p in paths.values()):
        out = {g: pd.read_parquet(p) for g, p in paths.items()}
        for g, t in out.items():
            if list(t["file"]) != list(df["file"]):
                raise ValueError(f"{paths[g]} does not match the requested clips")
        return out
    t0 = time.time()
    rows = Parallel(n_jobs=n_jobs, batch_size=32)(delayed(clip_features)(p, transform) for p in df["path"])
    root.mkdir(parents=True, exist_ok=True)
    out = {}
    for g in GROUPS:
        t = pd.DataFrame(np.stack([r[g] for r in rows]), columns=column_names(g))
        t.insert(0, "file", df["file"].to_numpy())
        t.to_parquet(paths[g])
        out[g] = t
    print(f"features {name}: {len(df)} clips in {time.time() - t0:.0f}s -> {root}/<group>_{name}.parquet")
    return out


def select(table, group, n_coef=N_CEP, deltas=2, stats=("mean", "std")):
    """Columns of a cached cepstral table for one ablation setting, e.g. 20 MFCC, no deltas."""
    keep = [f"{group}_d{d}_c{c:02d}_{s}" for d in range(deltas + 1) for c in range(n_coef)
            for s in stats]
    return table[keep]


SILENCE = ["lead_sil", "trail_sil", "sil_ratio", "duration", "energy_mean", "energy_std",
           "energy_p10", "energy_p90", "energy_max"]


def load_xy(split, groups=("lfcc",), n_coef=20, deltas=2, stats=("mean", "std"),
            root="data", variant=None):
    """One modelling table: file, label, speaker, attack + feature columns.

    `split` is "train", "dev" or "eval" (ASVspoof) or "itw" (In-the-Wild, read from
    <root>/release_in_the_wild/meta.csv). Groups follow the plan: "silence" (S), "mfcc" (M),
    "lfcc" (L), "spec" (P). Cepstral groups use `n_coef`, `deltas` and `stats`; the spectral group
    uses `stats`; silence uses all 9 columns. Feature columns are every column after the first four.
    `variant` (e.g. "trim") reads the features of transformed audio, <group>_<split>_<variant>.
    """
    root = Path(root)
    if split == "itw":
        from deepvoice.data import load_itw

        p = load_itw(root / "release_in_the_wild")
    else:
        p = pd.read_parquet(root / "protocols.parquet")
    out = p[p["split"] == split][["file", "label", "speaker", "attack"]].reset_index(drop=True)
    parts = [out]
    for g in groups:
        t = pd.read_parquet(root / "features" / f"{g}_{split}{'_' + variant if variant else ''}.parquet")
        if list(t["file"]) != list(out["file"]):
            raise ValueError(f"{g}_{split} rows do not match the protocol order")
        if g in ("mfcc", "lfcc"):
            parts.append(select(t, g, n_coef, deltas, stats))
        elif g == "spec":
            parts.append(t[[f"spec_{r}_{s}" for r in SPECTRAL for s in stats]])
        elif g == "silence":
            parts.append(t[SILENCE])
        else:
            raise ValueError(f"unknown feature group {g!r}")
    return pd.concat(parts, axis=1)
```

### `src/deepvoice/metrics.py`

```python
"""Evaluation metrics. Convention everywhere: label 1 = fake, 0 = real; higher score = more fake."""

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve


def _check(labels, other, name):
    labels = np.asarray(labels)
    other = np.asarray(other)
    if labels.shape != other.shape or labels.ndim != 1:
        raise ValueError(f"labels and {name} must be 1-D and the same length")
    if not set(np.unique(labels)) <= {0, 1}:
        raise ValueError("labels must be 0 (real) or 1 (fake)")
    return labels, other


def eer(labels, scores):
    """Equal error rate and the threshold where it occurs. Returns (eer, threshold).

    A clip is called fake when score >= threshold. The EER is read where the straight line
    between the two ROC points around the crossing meets fpr = fnr, so models with few distinct
    scores (e.g. a shallow tree) get a value for an operating point that exists; for continuous
    scores this equals the usual nearest-point estimate. The threshold is the nearer ROC point's.
    """
    labels, scores = _check(labels, scores, "scores")
    scores = scores.astype(float)
    if labels.min() == labels.max():
        raise ValueError("EER needs both classes present")
    # drop_intermediate=False: keep every threshold so the EER point is exact
    fpr, tpr, thr = roc_curve(labels, scores, drop_intermediate=False)
    fnr = 1 - tpr
    i = int(np.argmax(fnr <= fpr))  # first point past the crossing; point 0 has fpr=0, fnr=1
    d0, d1 = fpr[i - 1] - fnr[i - 1], fpr[i] - fnr[i]
    a = d0 / (d0 - d1)  # where the line between points i-1 and i crosses fpr = fnr
    value = fpr[i - 1] + a * (fpr[i] - fpr[i - 1])
    j = i if abs(d1) <= abs(d0) else i - 1
    return float(value), float(thr[j])


def speaker_bootstrap_eer(labels, scores, speakers, n_boot=1000, alpha=0.05, seed=0):
    """EER with a speaker-clustered bootstrap CI: resample whole speakers, not clips.

    Returns (point_eer, ci_low, ci_high). Resamples that lose a class are skipped.
    """
    labels, scores = _check(labels, scores, "scores")
    scores = scores.astype(float)
    speakers = np.asarray(speakers)
    if speakers.shape != labels.shape:
        raise ValueError("speakers must have the same length as labels")
    groups = pd.Series(np.arange(len(labels))).groupby(speakers).indices
    keys = list(groups)
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(keys), len(keys))
        idx = np.concatenate([groups[keys[k]] for k in pick])
        if labels[idx].min() == labels[idx].max():
            continue
        vals.append(eer(labels[idx], scores[idx])[0])
    if not vals:
        raise ValueError("every bootstrap resample lost a class; too few speakers?")
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return eer(labels, scores)[0], float(lo), float(hi)


def paired_speaker_bootstrap_diff(labels, scores_a, scores_b, speakers, n_boot=1000, alpha=0.05,
                                  seed=0):
    """EER(b) - EER(a) on the same clips, with a paired speaker-clustered bootstrap CI: each
    resample of whole speakers is scored with both a and b. Returns (diff, ci_low, ci_high)."""
    labels, scores_a = _check(labels, scores_a, "scores_a")
    _, scores_b = _check(labels, scores_b, "scores_b")
    speakers = np.asarray(speakers)
    groups = pd.Series(np.arange(len(labels))).groupby(speakers).indices
    keys = list(groups)
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[keys[k]] for k in rng.integers(0, len(keys), len(keys))])
        if labels[idx].min() == labels[idx].max():
            continue
        vals.append(eer(labels[idx], scores_b[idx])[0] - eer(labels[idx], scores_a[idx])[0])
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return eer(labels, scores_b)[0] - eer(labels, scores_a)[0], float(lo), float(hi)


def ece(labels, probs, n_bins=15):
    """Expected calibration error of P(fake): gap between mean predicted P(fake) and the
    observed fake rate, averaged over equal-width probability bins."""
    labels, probs = _check(labels, probs, "probs")
    labels = labels.astype(float)
    probs = probs.astype(float)
    if probs.min() < 0 or probs.max() > 1:
        raise ValueError("probs must be in [0, 1]; apply a sigmoid/softmax first")
    bins = np.minimum((probs * n_bins).astype(int), n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = bins == b
        if m.any():
            total += m.mean() * abs(probs[m].mean() - labels[m].mean())
    return float(total)


def per_attack_eer(labels, scores, attacks):
    """EER of each attack: that attack's fake clips against all real clips. Returns {attack: eer}."""
    labels, scores = _check(labels, scores, "scores")
    attacks = np.asarray(attacks)
    real = labels == 0
    return {a: eer(labels[real | (attacks == a)], scores[real | (attacks == a)])[0]
            for a in sorted(set(attacks[labels == 1]))}


def error_at_threshold(labels, scores, threshold):
    """Average of the false-accept rate (fakes scored below `threshold`, i.e. let through) and
    the false-reject rate (real clips scored >= `threshold`, i.e. flagged). A clip is called fake
    when score >= threshold, as in `eer`."""
    labels, scores = _check(labels, scores, "scores")
    fake = scores.astype(float) >= threshold
    far = float(np.mean(~fake[labels == 1]))
    frr = float(np.mean(fake[labels == 0]))
    return (far + frr) / 2


def _logit(probs):
    p = np.clip(np.asarray(probs, dtype=float), 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))


def fit_temperature(labels, probs):
    """Temperature T > 0 that minimizes the log loss of sigmoid(logit(p) / T) on (labels, probs).
    Fit it on dev; T > 1 softens over-confident probabilities."""
    from scipy.optimize import minimize_scalar

    labels, probs = _check(labels, probs, "probs")
    z, y = _logit(probs), labels.astype(float)

    def nll(log_t):
        q = np.clip(1 / (1 + np.exp(-z / np.exp(log_t))), 1e-12, 1 - 1e-12)
        return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))

    return float(np.exp(minimize_scalar(nll, bounds=(-5, 8), method="bounded").x))


def apply_temperature(probs, t):
    return 1 / (1 + np.exp(-_logit(probs) / t))


def reliability(labels, probs, n_bins=15):
    """Per equal-width bin: mean predicted P(fake), observed fake rate and clip count."""
    labels, probs = _check(labels, probs, "probs")
    bins = np.minimum((probs.astype(float) * n_bins).astype(int), n_bins - 1)
    rows = [(probs[bins == b].mean(), labels[bins == b].mean(), int((bins == b).sum()))
            for b in range(n_bins) if (bins == b).any()]
    return pd.DataFrame(rows, columns=["mean_prob", "frac_fake", "count"])
```

### `src/deepvoice/models.py`

```python
"""Model builders and the train/evaluate loop shared by rungs 0-2 (results CSV + run config)."""

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from deepvoice.metrics import speaker_bootstrap_eer

RESULT_COLUMNS = [
    "rung", "model", "config", "seed", "dataset", "split",
    "eer", "ci_low", "ci_high", "train_time", "latency",
]


def build(name, seed, **params):
    """A fresh sklearn model by name. Class weights are balanced (the data is ~90% fake).
    `params` override the defaults."""
    p = {"class_weight": "balanced", "random_state": seed, **params}
    if name == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(**{"max_iter": 2000, **p}))
    if name == "tree":
        return DecisionTreeClassifier(**p)
    if name == "rf":
        return RandomForestClassifier(**p)
    if name == "hgb":
        return HistGradientBoostingClassifier(**p)
    if name == "mlp":
        # sklearn's MLP has no class weights; EER is threshold-free, so imbalance matters less
        p.pop("class_weight")
        p = {"hidden_layer_sizes": (256,), "alpha": 1e-3, "early_stopping": True,
             "max_iter": 200, **p}
        return make_pipeline(StandardScaler(), MLPClassifier(**p))
    raise ValueError(f"unknown model {name!r}")


def run(rung, name, features, train, evals, seed=0, params=None, experiment=None,
        results_dir="results", n_boot=1000, feature_set=None):
    """Fit `name` on `train`, score every (dataset, split, frame) in `evals`, record results.

    `train` and each eval frame need the `features` columns plus `label` and `speaker`.
    Writes results/configs/<config>.json and appends one row per eval to
    results/<experiment>.csv. `feature_set` (e.g. "mfcc20_dd") is added to the model label and
    config name so runs of one model on different features stay apart.
    Returns (fitted model, list of result rows).
    """
    params = params or {}
    experiment = experiment or f"rung{rung}"
    label = f"{name}:{feature_set}" if feature_set else name
    config = f"{experiment}_{name}_{feature_set}_s{seed}" if feature_set else f"{experiment}_{name}_s{seed}"
    results_dir = Path(results_dir)
    (results_dir / "configs").mkdir(parents=True, exist_ok=True)
    cfg = {"rung": rung, "model": name, "feature_set": feature_set, "seed": seed, "params": params,
           "features": list(features),
           "train": f"{len(train)} clips", "n_boot": n_boot}
    (results_dir / "configs" / f"{config}.json").write_text(json.dumps(cfg, indent=2) + "\n")

    np.random.seed(seed)
    model = build(name, seed, **params)
    t0 = time.perf_counter()
    model.fit(train[features].to_numpy(), train["label"].to_numpy())
    train_time = time.perf_counter() - t0

    rows = []
    for dataset, split, df in evals:
        x = df[features].to_numpy()
        t0 = time.perf_counter()
        scores = model.predict_proba(x)[:, 1]  # P(fake): higher = more fake
        latency_ms = (time.perf_counter() - t0) / len(df) * 1000
        point, lo, hi = speaker_bootstrap_eer(
            df["label"].to_numpy(), scores, df["speaker"].to_numpy(), n_boot=n_boot, seed=seed
        )
        rows.append(dict(zip(RESULT_COLUMNS, [rung, label, config, seed, dataset, split,
                                              point, lo, hi, train_time, latency_ms])))
    out = results_dir / f"{experiment}.csv"
    new = pd.DataFrame(rows, columns=RESULT_COLUMNS)
    if out.exists():  # replace earlier rows of this exact config, keep the rest
        old = pd.read_csv(out)
        new = pd.concat([old[old["config"] != config], new], ignore_index=True)
    new.to_csv(out, index=False, float_format="%.6g")
    return model, rows


def fit_scores(name, params, train, features, frames, seed=0):
    """Fit on `train`; return P(fake) for each frame in `frames` and the fit time in seconds."""
    model = build(name, seed, **params)
    t0 = time.perf_counter()
    model.fit(train[features].to_numpy(), train["label"].to_numpy())
    fit_time = time.perf_counter() - t0
    return [model.predict_proba(f[features].to_numpy())[:, 1] for f in frames], fit_time


def grouped_cv_eer(name, params, train, features, seed=0, n_splits=5):
    """Mean EER over speaker-grouped folds of `train` (no speaker in both fit and test part)."""
    from sklearn.model_selection import GroupKFold

    from deepvoice.metrics import eer

    vals = []
    for fit_idx, test_idx in GroupKFold(n_splits=n_splits).split(train, groups=train["speaker"]):
        fit, test = train.iloc[fit_idx], train.iloc[test_idx]
        (s,), _ = fit_scores(name, params, fit, features, [test], seed)
        vals.append(eer(test["label"].to_numpy(), s)[0])
    return float(np.mean(vals))


def random_search(name, space, train, dev, features, n_iter=30, seed=0, fixed=None, cv=True):
    """Try `n_iter` settings drawn from `space`; score each on dev and by grouped CV on train.

    `fixed` params (e.g. n_jobs) are added to every setting. Returns one row per setting with
    the params (JSON), dev EER (point estimate), CV EER and fit time, sorted by dev EER.
    """
    from sklearn.model_selection import ParameterSampler

    from deepvoice.metrics import eer

    rows = []
    for i, p in enumerate(ParameterSampler(space, n_iter=n_iter, random_state=seed)):
        p = {**p, **(fixed or {})}
        (s,), fit_time = fit_scores(name, p, train, features, [dev], seed)
        row = {"model": name, "trial": i, "params": json.dumps(p, default=str),
               "dev_eer": eer(dev["label"].to_numpy(), s)[0], "fit_time": fit_time}
        if cv:
            row["cv_eer"] = grouped_cv_eer(name, p, train, features, seed)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("dev_eer").reset_index(drop=True)
```

### `src/deepvoice/pipeline.py`

```python
"""Build every cache the notebooks read, in order. Each step loads its cache if it exists, so
re-running is cheap; from scratch it takes about 3 hours on an Apple M5 (mostly XLS-R).

    uv run --extra dl python -m deepvoice.pipeline
"""

from pathlib import Path

import pandas as pd

from deepvoice.data import load_itw, load_protocols
from deepvoice.features import feature_tables
from deepvoice.silence import silence_table


def build_all(root="data", xlsr=True):
    root = Path(root)
    proto_file = root / "protocols.parquet"
    if not proto_file.exists():
        load_protocols(root / "LA").to_parquet(proto_file)
    p = pd.read_parquet(proto_file)
    sets = {s: p[p.split == s].reset_index(drop=True) for s in ["train", "dev", "eval"]}
    sets["itw"] = load_itw(root / "release_in_the_wild").reset_index(drop=True)
    feats = root / "features"
    for s, df in sets.items():
        silence_table(df, feats / f"silence_{s}.parquet")
        feature_tables(df, s, root=feats)
    for change in ["trim", "gain-12", "gain+6"]:  # E5 test-time changes
        for s in ["eval", "itw"]:
            feature_tables(sets[s], f"{s}_{change}", root=feats, transform=change)
    feature_tables(sets["train"], "train_augment", root=feats, transform="augment")  # E5 mitigation
    if xlsr:
        from deepvoice import xlsr as xl

        model = None
        jobs = [(s, None) for s in sets] + [("eval_trim", "trim"), ("itw_trim", "trim")]
        for name, change in jobs:
            if not xl.cache_path(name, feats).exists():
                model = model or xl.load_model()
            xl.extract_cached(sets[name.split("_")[0]], name, root=feats, model=model,
                              transform=change)


if __name__ == "__main__":
    build_all()
    print("all caches ready")
```

### `src/deepvoice/predict.py`

```python
"""Score audio files with the final DeepVoice model (rung 2: frozen XLS-R-300M layer 3 + logistic
regression).

    uv run --extra dl python -m deepvoice.predict clip1.wav clip2.flac

Prints P(fake) and the decision at the ASVspoof-dev EER threshold. Read model/MODEL_CARD.md
first: on real-world audio (In-the-Wild) this model is close to chance.
"""

import json
import sys
from pathlib import Path

import joblib
import numpy as np

MODEL_DIR = Path(__file__).resolve().parents[2] / "model"
MODEL_FILE = "rung2_xlsr300m_l3_logreg.joblib"


def load(model_dir=MODEL_DIR):
    """(classifier, metadata) of the saved final model."""
    model_dir = Path(model_dir)
    meta = json.loads((model_dir / MODEL_FILE.replace(".joblib", ".json")).read_text())
    return joblib.load(model_dir / MODEL_FILE), meta


def score(paths, clf=None, meta=None, xlsr_model=None):
    """P(fake) for each audio file (any sample rate; resampled to 16 kHz mono)."""
    from deepvoice import xlsr

    if clf is None:
        clf, meta = load()
    xlsr_model = xlsr_model or xlsr.load_model()
    emb = xlsr.embed_paths(xlsr_model, [str(p) for p in paths])
    x = emb[:, xlsr.LAYERS.index(meta["layer"])].astype(np.float32)
    return clf.predict_proba(x)[:, 1]


def main(argv=None):
    paths = argv if argv is not None else sys.argv[1:]
    if not paths:
        print(__doc__)
        return 2
    clf, meta = load()
    for p, s in zip(paths, score(paths, clf, meta)):
        label = "fake" if s >= meta["threshold"] else "real"
        print(f"{p}\tP(fake)={s:.3f}\t{label}")
    return 0


if __name__ == "__main__":
    sys.exit(main())
```

### `src/deepvoice/silence.py`

```python
"""Silence and duration features (group S): leading/trailing silence, silence ratio, duration and
frame-energy statistics from an energy-based VAD.

VAD rule: a frame is speech if its energy is within REL_DB of the clip's loudest frame. REL_DB is
a fixed constant chosen before looking at any data, so it cannot leak dev/eval information.
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from deepvoice.features import SR, load_audio

FRAME = 400  # 25 ms at 16 kHz
HOP = 160  # 10 ms
REL_DB = 35.0
COLUMNS = [
    "duration",
    "lead_sil",
    "trail_sil",
    "sil_ratio",
    "energy_mean",
    "energy_std",
    "energy_p10",
    "energy_p90",
    "energy_max",
]


def frame_energy_db(y):
    """RMS energy of each 25 ms frame (10 ms hop), in dB relative to full scale."""
    if len(y) < FRAME:
        y = np.pad(y, (0, FRAME - len(y)))
    frames = np.lib.stride_tricks.sliding_window_view(y, FRAME)[::HOP]
    rms = np.sqrt((frames.astype(np.float64) ** 2).mean(axis=1))
    return 20 * np.log10(rms + 1e-10)


def silence_features(y, sr=SR):
    """Group S features of one clip (seconds for times, dBFS for energies)."""
    e = frame_energy_db(y)
    speech = e >= e.max() - REL_DB
    first = int(np.argmax(speech))
    last = len(speech) - 1 - int(np.argmax(speech[::-1]))
    hop_s = HOP / sr
    return {
        "duration": len(y) / sr,
        "lead_sil": first * hop_s,
        "trail_sil": (len(speech) - 1 - last) * hop_s,
        "sil_ratio": float(1 - speech.mean()),
        "energy_mean": float(e.mean()),
        "energy_std": float(e.std()),
        "energy_p10": float(np.percentile(e, 10)),
        "energy_p90": float(np.percentile(e, 90)),
        "energy_max": float(e.max()),
    }


def trim(y, sr=SR):
    """Cut leading and trailing silence (same VAD rule as the features); keeps the speech span."""
    e = frame_energy_db(y)
    speech = e >= e.max() - REL_DB
    first = int(np.argmax(speech))
    last = len(speech) - 1 - int(np.argmax(speech[::-1]))
    return y[first * HOP: min(len(y), last * HOP + FRAME)]


def _one(path):
    return silence_features(load_audio(path))


def silence_table(df, cache, n_jobs=-1):
    """Silence features for every row of a protocol table, cached at `cache` (parquet).

    If the cache exists it is loaded, never recomputed. Rows keep the order of `df["file"]`.
    """
    cache = Path(cache)
    if cache.exists():
        out = pd.read_parquet(cache)
        if list(out["file"]) != list(df["file"]):
            raise ValueError(f"{cache} does not match the requested clips; delete it on purpose")
        return out
    t0 = time.time()
    rows = Parallel(n_jobs=n_jobs, batch_size=64)(delayed(_one)(p) for p in df["path"])
    out = pd.concat([df[["file"]].reset_index(drop=True), pd.DataFrame(rows)], axis=1)
    cache.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(cache)
    print(f"silence features: {len(out)} clips in {time.time() - t0:.0f}s -> {cache}")
    return out
```

### `src/deepvoice/xlsr.py`

```python
"""Frozen XLS-R-300M embeddings (rung 2): mean-pooled hidden states every 3rd layer, fp16, cached.

The model is never trained here. Each clip goes through it once; for layers 0, 3, ..., 24 we
average the frame vectors over time (ignoring padding) and keep that 1024-dim vector.
"""

import time
from pathlib import Path

import numpy as np
import torch

from deepvoice.features import apply_transform, load_audio

MODEL = "facebook/wav2vec2-xls-r-300m"
LAYERS = list(range(0, 25, 3))  # 0 = CNN feature projection output, 24 = last transformer layer


def device():
    return torch.device("mps" if torch.backends.mps.is_available() else "cpu")


def load_model(dev=None):
    from transformers import Wav2Vec2Model

    model = Wav2Vec2Model.from_pretrained(MODEL)
    return model.eval().to(dev or device())


def _normalize(y):
    # what Wav2Vec2FeatureExtractor does: zero mean, unit variance per clip
    return (y - y.mean()) / np.sqrt(y.var() + 1e-7)


@torch.inference_mode()
def embed_batch(model, waves, dev=None):
    """Mean-pooled embeddings for a list of 16 kHz clips -> float16 array (n, len(LAYERS), 1024)."""
    dev = dev or next(model.parameters()).device
    lens = [len(w) for w in waves]
    x = np.zeros((len(waves), max(lens)), dtype=np.float32)
    mask = np.zeros_like(x, dtype=np.int64)
    for i, w in enumerate(waves):
        x[i, : len(w)] = _normalize(w)
        mask[i, : len(w)] = 1
    out = model(torch.from_numpy(x).to(dev), attention_mask=torch.from_numpy(mask).to(dev),
                output_hidden_states=True)
    frames = model._get_feat_extract_output_lengths(torch.tensor(lens)).to(dev)
    fmask = (torch.arange(out.hidden_states[0].shape[1], device=dev)[None] < frames[:, None])
    fmask = fmask[..., None].float()
    pooled = [(out.hidden_states[k] * fmask).sum(1) / fmask.sum(1) for k in LAYERS]
    return torch.stack(pooled, 1).half().cpu().numpy()


def embed_paths(model, paths, batch_seconds=60.0, transform=None):
    """Embeddings for many files. Clips are sorted by length and batched up to ~batch_seconds of
    audio per batch (less padding, fewer passes); output keeps the input order."""
    waves = [apply_transform(load_audio(p), transform) for p in paths]
    order = np.argsort([len(w) for w in waves])
    out = np.zeros((len(waves), len(LAYERS), 1024), dtype=np.float16)
    batch = []
    for i in order:
        batch.append(i)
        if len(waves[i]) * len(batch) / 16000 >= batch_seconds or i == order[-1]:
            out[batch] = embed_batch(model, [waves[j] for j in batch])
            batch = []
    return out


def time_it(paths, model=None, batch_seconds=60.0):
    """Seconds per clip and per audio-second for embedding `paths` (model load excluded)."""
    model = model or load_model()
    embed_paths(model, paths[:4], batch_seconds)  # warm-up: first mps call compiles kernels
    if model.device.type == "mps":
        torch.mps.synchronize()
    t0 = time.perf_counter()
    emb = embed_paths(model, paths, batch_seconds)
    if model.device.type == "mps":
        torch.mps.synchronize()
    dt = time.perf_counter() - t0
    audio_s = sum(len(load_audio(p)) for p in paths) / 16000
    return {"clips": len(paths), "seconds": dt, "s_per_clip": dt / len(paths),
            "audio_seconds": audio_s, "realtime_factor": audio_s / dt, "emb": emb}


def cache_path(name, root="data/features"):
    return Path(root) / f"xlsr300m_{name}.npy"


def extract_cached(df, name, root="data/features", chunk=2000, model=None, transform=None):
    """Embeddings for every row of `df` -> <root>/xlsr300m_<name>.npy (n, 9 layers, 1024), fp16,
    rows in `df` order, plus <name>_files.txt listing the clip ids. Work is saved per chunk in
    <root>/xlsr300m_<name>_parts/, so an interrupted run resumes where it stopped."""
    out = cache_path(name, root)
    files = out.with_name(f"xlsr300m_{name}_files.txt")
    if out.exists():
        if files.read_text().split() != list(df["file"]):
            raise ValueError(f"{out} does not match the requested clips")
        return np.load(out, mmap_mode="r")
    parts = out.with_name(f"xlsr300m_{name}_parts")
    parts.mkdir(parents=True, exist_ok=True)
    paths = list(df["path"])
    n_chunks = (len(paths) + chunk - 1) // chunk
    t0 = time.time()
    for c in range(n_chunks):
        part = parts / f"{c:04d}.npy"
        if part.exists():
            continue
        model = model or load_model()
        np.save(part, embed_paths(model, paths[c * chunk:(c + 1) * chunk], transform=transform))
        print(f"{name}: chunk {c + 1}/{n_chunks} done, {time.time() - t0:.0f}s", flush=True)
    emb = np.concatenate([np.load(parts / f"{c:04d}.npy") for c in range(n_chunks)])
    np.save(out, emb)
    files.write_text("\n".join(df["file"]) + "\n")
    return emb
```